# HHMARL 2D in JAX: massively parallel reproduction on a single Kaggle GPU

This notebook is a self-contained JAX re-implementation of **IDSIA/hhmarl_2D** (Selmonaj et al.,
*Hierarchical Multi-Agent Reinforcement Learning for Air Combat Maneuvering*, arXiv 2309.11247, ICMLA 2023).
It replaces the original Python simulator and the Ray RLlib PPO setup with:

* a fully `jit` + `vmap` compiled environment (low-level fight/escape env with the L1–L5 curriculum,
  scripted and fictitious-self-play opponents, heterogeneous AC1/AC2 aircraft; and the hierarchical
  commander env that calls frozen low-level policies), so thousands of combats run in parallel on one GPU;
* the original networks (self-attention fight nets, escape nets, GRU commander, centralized critics) and
  PPO with the original RLlib hyper-parameters, including (behind `RLLIB_FAITHFUL`) the RLlib training quirks;
* training runs for the paper experiments (Fig. 6, 8, 10), FSP levels L4/L5, evaluation (Fig. 7, 8b, 10b)
  and paper-style figures with the paper's numbers overlaid.

**Defaults follow the original code, not the paper.** Paper-vs-code differences are switches in the config cell.

How to use on Kaggle: enable a GPU (T4 x1 recommended), choose `PRESET`, `BUDGET` and `EXPERIMENTS` in the
config cell, then *Run All*. Every stage writes checkpoints, CSV logs and figures to `OUT_DIR`. A 12 h session
limit is handled by `MAX_SESSION_HOURS`: the notebook checkpoints and stops cleanly; attach the output of the
previous run as an input dataset and run again to continue (see `RESUME_DIRS`).

Sections: 1 Setup · 2 Config · 3 JAX env · 4 Networks + PPO · 5 Training runs · 6 Evaluation ·
7 Figures · 8 Known deviations.

## 1. Setup
Installs JAX with CUDA 12 wheels on Kaggle (skipped locally or when `HHMARL_SKIP_PIP=1`), checks the
accelerator and creates the output folders. `OUT_DIR` defaults to `/kaggle/working` and falls back to
`./outputs` when not on Kaggle (override with the env var `HHMARL_OUT_DIR`).

In [ ]:
import os, sys, subprocess, time, json, glob, shutil, pickle, math, csv, functools
IN_KAGGLE = os.path.isdir('/kaggle/working')
if IN_KAGGLE and os.environ.get('HHMARL_SKIP_PIP', '0') != '1':
    # jax[cuda12] brings its own CUDA/cuDNN wheels; optax for Adam; pillow for GIF export.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'jax[cuda12]', 'optax',
                    'matplotlib', 'pillow'], check=False)

import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('python', sys.version.split()[0], '| jax', jax.__version__, '| optax', optax.__version__)
print('devices:', jax.devices())
BACKEND = jax.default_backend()
if BACKEND != 'gpu':
    print('WARNING: no GPU found, running on', BACKEND, '(fine for SMOKE runs, far too slow for full training)')
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
except Exception:
    pass

OUT_DIR = os.environ.get('HHMARL_OUT_DIR') or ('/kaggle/working' if IN_KAGGLE else os.path.abspath('./outputs'))
DIRS = {k: os.path.join(OUT_DIR, k) for k in ['checkpoints', 'policies', 'logs', 'figures', 'eval']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
SESSION_START = time.time()
print('OUT_DIR =', OUT_DIR)

## 2. Configuration

* **`SMOKE`**: tiny CPU-sized run of every stage (a few hundred env steps each). Keep `False` for real runs.
* **`PRESET`**:
  * `faithful`: the original RLlib batch geometry. 2000 env steps per PPO update for the low level and
    1000 for the commander, minibatch 256, 30 SGD epochs, few envs. Slow on a GPU because it is sequential
    SGD-bound, but it reproduces the original sample reuse.
  * `throughput`: about 1000 parallel envs and large batches, for GPU speed. It has fewer gradient steps per env step, so
    learning curves per env step will differ from the paper.
* **`BUDGET`**: env steps per stage (`demo` about 1 GPU-hour in total, `full` at about the paper's scale; see the table).
  `BUDGET_SCALE` multiplies all stage budgets.
* **`EXPERIMENTS`**: which parts to run. Stages that are already finished in `OUT_DIR` (or in a resumed input
  dataset) are skipped automatically.
* **Paper-vs-code switches**: every default is the ORIGINAL CODE value. Set a `PAPER_*` switch to `True` to use the
  value stated in the paper instead.
* **`RLLIB_FAITHFUL`**: replicates the RLlib training quirks documented in the spec. With `True`:
  * the fight self-attention runs over ≤ 20-step chunks *including future steps* and zero-padded rows during
    SGD, while rollouts use a length-1 sequence;
  * the unmasked loss includes the padded rows;
  * the centralized critic sees zero actions at rollout/GAE time and the real actions during training;
  * the commander GRU state is reset each step and chained across agents at evaluation.

  With `False`, the clean versions are used.

In [ ]:
SMOKE = False                      # tiny end-to-end check (CPU friendly). Keep False for real runs.
if os.environ.get('HHMARL_SMOKE') is not None:
    SMOKE = os.environ['HHMARL_SMOKE'] == '1'

PRESET = 'faithful'                # 'faithful' | 'throughput'
BUDGET = 'demo'                    # 'demo' | 'full'
BUDGET_SCALE = 1.0                 # multiplies every stage budget
EXPERIMENTS = ['fig6', 'fig8', 'fsp', 'fig10', 'eval', 'figures']   # subset of these, in this order
SEED = 0
MAX_SESSION_HOURS = 11.3           # stop training cleanly (checkpoint) before Kaggle's 12 h limit
CKPT_EVERY_MIN = 20                # periodic checkpoint inside a stage (minutes)
# Previous outputs to resume from (attach a previous run's /kaggle/working as an input dataset).
RESUME_DIRS = sorted(glob.glob('/kaggle/input/*')) + sorted(glob.glob('/kaggle/input/*/*'))

# ---- RLlib training quirks -------------------------------------------------------------------
RLLIB_FAITHFUL = True

# ---- paper-vs-code switches (False = original code, the default) ----------------------------
PAPER_PPO = False          # gamma 0.95 / clip 0.2 (paper) instead of gamma 0.99 / clip 0.25 (code)
PAPER_HORIZONS = False     # L1..L5 horizons 200/250/300/350/400 (paper) instead of 150/200/300/350/400
PAPER_HIT_PROB = False     # cannon hit prob per burst 0.70/0.85 (paper) instead of 0.75/0.90
PAPER_HL_OOB = False       # commander out-of-bounds penalty -5 (paper) instead of -2
PAPER_L5_POOL = False      # L5 opponents = L1..L4 fight policies (paper) instead of {L3 fight, L4 fight, L3 escape}
ESC_DIST_REW = False       # code flag `esc_dist_rew` (per-step distance reward for escape; default off)
FRIENDLY_KILL = True       # code default: cannon/rockets can hit teammates
FRIENDLY_PUNISH = False    # code default: victim of friendly fire not punished
HIER_OPP_FIGHT_RATIO = 75  # % of commander-env opponents choosing the fight policy
HIER_ACTION_ASSESS = True  # commander situation reward (+0.1 / -0.1)
HL_OPP_START_SPEED0 = True # code quirk: args.level defaults to 1 in commander mode -> opponents start at speed 0
ESCAPE_L5_STAGE = False    # optional README step: train escape again at L5 vs L5 fight
FIG6_FC_CURRICULUM = False # FC baseline of Fig. 6 trained L1->L3 (True) or at L3 from scratch (False)

# ---- PPO hyper-parameters (train_hetero.py / train_hier.py + RLlib 2.4 defaults) -------------
PPO_LOW = dict(lr=1e-4, gamma=0.99, lam=0.95, clip=0.25, kl_coeff=0.2, kl_target=0.025,
               vf_coeff=1.0, vf_clip=10.0, ent_coeff=0.0, max_seq_len=20)
PPO_CMD = dict(lr=1e-4, gamma=0.99, lam=1.0, clip=0.25, kl_coeff=0.2, kl_target=0.05,
               vf_coeff=1.0, vf_clip=10.0, ent_coeff=0.0, max_seq_len=20)
if PAPER_PPO:
    for _p in (PPO_LOW, PPO_CMD):
        _p.update(gamma=0.95, clip=0.2)

PRESETS = {
    # n_envs * rollout = env steps per PPO update (RLlib: 2000 low level, 1000 commander)
    'faithful':   dict(low_envs=10, low_rollout=200, low_minibatch=256, low_epochs=30,
                       cmd_envs=10, cmd_rollout=100, cmd_minibatch=256, cmd_epochs=30),
    'throughput': dict(low_envs=1024, low_rollout=40, low_minibatch=4096, low_epochs=10,
                       cmd_envs=256, cmd_rollout=20, cmd_minibatch=1536, cmd_epochs=10),
}
HP = dict(PRESETS[PRESET])

# env steps per stage (commander: commander steps, as on the paper's Fig. 10 x-axis)
BUDGETS = {
    'demo': dict(fight_L1=1.5e6, fight_L2=1.5e6, fight_L3=3e6, fight_L3_nocurr=3e6, fight_L3_fc=3e6,
                 escape_L3=1.5e6, fight_L4=1.5e6, fight_L5=1.5e6, escape_L5=1e6,
                 commander_gru=0.5e6, commander_sa=0.5e6, commander_fc=0.5e6),
    # about 10k RLlib iterations per level (code default), L3/escape/commander lengths read off the paper axes
    'full': dict(fight_L1=20e6, fight_L2=20e6, fight_L3=28e6, fight_L3_nocurr=28e6, fight_L3_fc=28e6,
                 escape_L3=8.2e6, fight_L4=20e6, fight_L5=20e6, escape_L5=8e6,
                 commander_gru=6.7e6, commander_sa=6.7e6, commander_fc=6.7e6),
}
STAGE_BUDGET = {k: int(v * BUDGET_SCALE) for k, v in BUDGETS[BUDGET].items()}

EVAL_EPISODES = 1000               # paper: 1000 episodes per scenario
EVAL_SIZES = [2, 3, 4, 5]          # n-vs-n evaluation scenarios

if SMOKE:
    HP = dict(low_envs=4, low_rollout=40, low_minibatch=40, low_epochs=2,
              cmd_envs=3, cmd_rollout=20, cmd_minibatch=40, cmd_epochs=2)
    STAGE_BUDGET = {k: (160 if not k.startswith('commander') else 60) for k in STAGE_BUDGET}
    EVAL_EPISODES = 3
    CKPT_EVERY_MIN = 0.01

HORIZON_LOW = {1: 150, 2: 200, 3: 300, 4: 350, 5: 400}
if PAPER_HORIZONS:
    HORIZON_LOW = {1: 200, 2: 250, 3: 300, 4: 350, 5: 400}
HORIZON_HL = 500
MAP_LOW, MAP_HL = 0.3, 0.5         # degrees (about 33 km / 55 km)
HIT_PROB = (0.70, 0.85) if PAPER_HIT_PROB else (0.75, 0.90)

CONFIG = dict(SMOKE=SMOKE, PRESET=PRESET, BUDGET=BUDGET, HP=HP, STAGE_BUDGET=STAGE_BUDGET, SEED=SEED,
              RLLIB_FAITHFUL=RLLIB_FAITHFUL, PAPER_PPO=PAPER_PPO, PAPER_HORIZONS=PAPER_HORIZONS,
              PAPER_HIT_PROB=PAPER_HIT_PROB, PAPER_HL_OOB=PAPER_HL_OOB, PAPER_L5_POOL=PAPER_L5_POOL,
              ESC_DIST_REW=ESC_DIST_REW, FRIENDLY_KILL=FRIENDLY_KILL, FRIENDLY_PUNISH=FRIENDLY_PUNISH,
              HIER_OPP_FIGHT_RATIO=HIER_OPP_FIGHT_RATIO, HIER_ACTION_ASSESS=HIER_ACTION_ASSESS,
              HL_OPP_START_SPEED0=HL_OPP_START_SPEED0, PPO_LOW=PPO_LOW, PPO_CMD=PPO_CMD,
              EVAL_EPISODES=EVAL_EPISODES, EXPERIMENTS=EXPERIMENTS)
with open(os.path.join(OUT_DIR, 'config.json'), 'w') as _f:
    json.dump(CONFIG, _f, indent=1)
print(json.dumps(CONFIG, indent=1))

## 3. The JAX environment

One `Env` class covers both original environments:

* `kind='low'` corresponds to `LowLevelEnv` (2 vs 2, 30 km map). It runs fight or escape mode and curriculum levels 1–5:
  * L1 static, L2 random and L3 scripted opponents;
  * L4 and L5 fictitious self-play opponents driven by **frozen policy parameters** (deterministic argmax, like `_policy_actions`).
* `kind='high'` corresponds to `HighLevelEnv` (N vs M up to 5 vs 5, 50 km map). One `step` is a commander step:
  * opponent fight/escape assignment;
  * then up to 16 low-level ticks, in which every aircraft is driven by frozen low-level fight or escape policies;
  * the commander step exits early on any kill/out-of-bounds event, or on a close engagement after tick 12.

Units are fixed slots `0..N-1` (agents, blue) and `N..N+M-1` (opponents, red); dead units are masked
(`alive`). Every function is pure and written for a single env; batching is `jax.vmap`.

Faithfulness notes (see `docs/spec_original_env.md`), all reproduced:
* **Tick order:**
  * aircraft update in id order, then rockets in launch order;
  * units killed earlier in the same tick still act (fire, steer, move);
  * the cannon kills immediately, with a Bernoulli p_hit/burst per target per tick;
  * the cannon cone uses the post-turn heading and the pre-move position.
* **Weapon quirks:**
  * rockets are unguided, with multiplicative heading noise clip(h·U(0.95,1.05), 0, 359);
  * the radar cone is [h, h+120];
  * the cooldown is set even when a launch fails;
  * scripted opponents never count their cooldown down;
  * every rocket's proximity fuse is hard-coded to aircraft id 2;
  * a rocket hit is checked before the rocket moves.
* **Observation quirks:**
  * headings are normalized by 359;
  * observation and reward geometry is planar in degrees, weapon geometry is geodesic (local ellipsoidal flat-earth);
  * FSP opponents observe the agents' *current-step* shot flags;
  * the commander's low-level observations use distances frozen at the start of the commander step;
  * an escaping aircraft's missile targets the farthest opponent.
* **Reward and episode quirks:**
  * kill rewards earned by an agent that died earlier are dropped;
  * dead agents keep emitting zero observations;
  * the time limit is a true terminal.

In [ ]:
KN2MS = 0.514444
_WGS_A = 6378137.0
_WGS_F = 1.0 / 298.257223563
_WGS_E2 = _WGS_F * (2.0 - _WGS_F)
LAT0 = 5.0                    # map origin (lat 5, lon 7); positions are stored as offsets in degrees
D2R = math.pi / 180.0
# rocket speed profile interp1d([0,10,20,30],[500,2000,1400,600],'quadratic') at integer life 0..10 s
ROCKET_SPEED = np.array([500.0, 758.75, 993.3333333, 1203.75, 1390.0, 1552.0833333, 1690.0, 1803.75,
                         1893.3333333, 1958.75, 2000.0], np.float32)
SPLITS1, SPLITS2 = (13, 9, 2, 2), (13, 9, 2)        # MultiDiscrete action spaces of AC1 / AC2
OBS_FIGHT = (26, 24)
OBS_ESC = (30, 29)
OBS_CMD = 34


def _radii(lat_abs):
    s = jnp.sin(lat_abs * D2R)
    w = jnp.sqrt(1.0 - _WGS_E2 * s * s)
    return _WGS_A * (1.0 - _WGS_E2) / w ** 3, _WGS_A / w        # meridional M, normal N


def geo_inv(lat1, lon1, lat2, lon2):
    """Geodesic distance [km] and initial bearing [deg, 0..360) (local ellipsoidal approximation,
    < 0.2 m / < 0.001 deg error on these maps vs geographiclib)."""
    lm = LAT0 + 0.5 * (lat1 + lat2)
    M, N = _radii(lm)
    dn = (lat2 - lat1) * D2R * M
    de = (lon2 - lon1) * D2R * N * jnp.cos(lm * D2R)
    dist = jnp.sqrt(dn * dn + de * de) / 1000.0
    brg = jnp.degrees(jnp.arctan2(de, dn)) - (lon2 - lon1) * jnp.sin(lm * D2R) / 2.0
    return dist, jnp.mod(brg, 360.0)


def geo_move(lat, lon, hdg, dist_m):
    """Geodesic direct step (midpoint flat-earth, < 1 cm error per 1 km step)."""
    c, s = jnp.cos(hdg * D2R), jnp.sin(hdg * D2R)
    M, _ = _radii(LAT0 + lat)
    lm = LAT0 + lat + 0.5 * dist_m * c / M / D2R
    M, N = _radii(lm)
    return lat + dist_m * c / M / D2R, lon + dist_m * s / (N * jnp.cos(lm * D2R)) / D2R


def shd(actual, desired):
    """warsim signed_heading_diff."""
    d = desired - actual
    d = jnp.where(d < -180.0, d + 360.0, d)
    return jnp.where(d > 180.0, d - 360.0, d)


def tree_where(c, a, b):
    return jax.tree_util.tree_map(lambda x, y: jnp.where(jnp.reshape(c, c.shape + (1,) * (x.ndim - c.ndim)), x, y), a, b)


def _uni(k, lo, hi, shape=()):
    return jax.random.uniform(k, shape, minval=lo, maxval=hi)


def _randint(k, lo, hi, shape=()):          # inclusive bounds like python random.randint
    return jax.random.randint(k, shape, lo, hi + 1)


class Env:
    """kind='low': LowLevelEnv (2v2). kind='high': HighLevelEnv (NvM, commander steps)."""

    def __init__(self, kind='low', n=2, m=2, level=3, mode='fight', opp_pool=None, record=False):
        self.kind, self.hl = kind, kind == 'high'
        self.N, self.M, self.T, self.K = n, m, n + m, max(n, m)
        self.level, self.mode, self.record = level, mode, record
        self.ms = MAP_HL if self.hl else MAP_LOW
        self.horizon = HORIZON_HL if self.hl else HORIZON_LOW[level]
        self.dnorm = math.sqrt(2 * self.ms ** 2)
        self.team = np.array([0] * n + [1] * m)
        self.is_ag = jnp.asarray(self.team == 0)
        if not self.hl:
            assert n == 2 and m == 2, 'the original low-level env is 2 vs 2 (fixed partners 1<->2, 3<->4)'
            self.partner = jnp.array([1, 0, 3, 2])
            if level == 4:
                opp_pool = opp_pool or ('fight',)
            elif level == 5:
                opp_pool = opp_pool or ((('fight',) * 4) if PAPER_L5_POOL else ('fight', 'fight', 'escape'))
            self.opp_pool = tuple(opp_pool or ())
        self.obs_dim = OBS_CMD if self.hl else (26 if mode == 'fight' else 30)
        self.p_tick = (HIT_PROB[0] / 5.0, HIT_PROB[1] / 3.0)
        self.oob_pen = (-5.0 if PAPER_HL_OOB else -2.0) if self.hl else -5.0

    # ------------------------------------------------------------------ reset
    def reset(self, key):
        N, M, T = self.N, self.M, self.T
        ks = jax.random.split(key, 8)
        r = _randint(ks[0], 1, 2)
        team = jnp.asarray(self.team)
        idx = jnp.asarray(np.r_[np.arange(N), np.arange(M)]).astype(jnp.float32)
        u = jax.random.uniform(ks[1], (T, 2))
        ag = team == 0
        left = jnp.where(ag, r == 1, r == 2)          # this unit spawns on the west side
        L = self.level
        if self.hl:
            K = jnp.where(ag, float(N), float(M))
            lon = jnp.where(left, 0.07 + 0.15 * u[:, 0], 0.28 + 0.15 * u[:, 0])
            lat = 0.07 + idx * 0.4 / K + 0.05 * u[:, 1]
            hdg = _randint(ks[2], 0, 359, (T,)).astype(jnp.float32)
        else:
            if L == 1:
                lon = jnp.where(left, 0.12 + 0.02 * u[:, 0], 0.16 + 0.01 * u[:, 0])
                lat = 0.10 + 0.1 * idx + 0.01 * u[:, 1]
                h_ag = jnp.where(r == 1, _randint(ks[2], 30, 150, (T,)), _randint(ks[3], 200, 330, (T,)))
                hdg = jnp.where(ag, h_ag, 0).astype(jnp.float32)
            elif L == 2:
                lon = jnp.where(left, 0.08 + 0.05 * u[:, 0], 0.18 + 0.05 * u[:, 0])
                lat = 0.08 + 0.1 * idx + 0.05 * u[:, 1]
                h_ag = jnp.where(r == 1, _randint(ks[2], 0, 180, (T,)), _randint(ks[3], 180, 359, (T,)))
                hdg = jnp.where(ag, h_ag, _randint(ks[4], 0, 359, (T,))).astype(jnp.float32)
            else:
                lon = jnp.where(left, 0.07 + 0.05 * u[:, 0], 0.18 + 0.05 * u[:, 0])
                lat = 0.09 + 0.1 * idx + 0.03 * u[:, 1]
                h_ag = jnp.where(r == 1, _randint(ks[2], 0, 270, (T,)), _randint(ks[3], 90, 359, (T,)))
                hdg = jnp.where(ag, h_ag, _randint(ks[4], 0, 359, (T,))).astype(jnp.float32)
        slot = jnp.asarray(np.r_[np.arange(N), np.arange(M)])
        ac1 = jnp.where(slot == 0, True, jnp.where(slot == 1, False, jax.random.bernoulli(ks[5], 0.5, (T,))))
        opp_slow = (~ag) & ((L <= 2) if not self.hl else HL_OPP_START_SPEED0)
        spd = jnp.where(opp_slow, 0.0, 100.0)
        if self.hl:
            can = jnp.full((T,), 300.0); mis = jnp.where(ac1, 8.0, 0.0)
        elif L == 5:
            can = jnp.full((T,), 300.0); mis = jnp.where(ac1, 6.0, 0.0)
        else:
            can = jnp.where(ag, 200.0, 400.0); mis = jnp.where(ac1, jnp.where(ag, 5.0, 8.0), 0.0)
        f0, i0, b0 = jnp.zeros((T,), jnp.float32), jnp.zeros((T,), jnp.int32), jnp.zeros((T,), bool)
        s = dict(lat=lat.astype(jnp.float32), lon=lon.astype(jnp.float32), hdg=hdg, nhdg=hdg, spd=spd, nspd=spd,
                 ac1=ac1, alive=jnp.ones((T,), bool), can_rem=can, can_max=can, burst=f0,
                 mis_rem=mis, mis_max=mis, has_mis=b0, mwait=i0,
                 r_alive=b0, r_lat=f0, r_lon=f0, r_hdg=f0, r_nhdg=f0, r_spd=f0, r_tgt=i0, r_t0=i0, r_seq=i0,
                 seq_ctr=jnp.int32(0), t=jnp.int32(0), steps=jnp.int32(0),
                 esc_flag=jnp.array(False), esc_timer=jnp.int32(0), tgt=jnp.full((T,), -1, jnp.int32),
                 opp_k=jnp.int32(0), ca=jnp.zeros((T,), jnp.int32),
                 lst=jnp.full((T, self.K), -1, jnp.int32), lst_dn=jnp.zeros((T, self.K)), lst_n=i0,
                 # episode statistics for evaluation: kills/deaths/friendly kills per agent aircraft type
                 st_k=jnp.zeros(2), st_d=jnp.zeros(2), st_fk=jnp.zeros(2), st_ag_by_opp=jnp.float32(0),
                 st_opp_by_ag=jnp.float32(0), st_oob=jnp.float32(0))
        if not self.hl and L == 5:
            s['opp_k'] = jax.random.randint(ks[6], (), 0, len(self.opp_pool))
        if self.hl:
            obs, s = self._cmd_obs(s)
        else:
            obs, s = self._low_obs(s)
        return s, obs

    # ------------------------------------------------------------------ geometry & observations
    def _planar(self, s):
        dx = s['lon'][None, :] - s['lon'][:, None]
        dy = s['lat'][None, :] - s['lat'][:, None]
        dd = jnp.sqrt(dx * dx + dy * dy)                                    # [i,j] planar distance (deg)
        hr = s['hdg'] * D2R
        ux, uy = jnp.sin(hr), jnp.cos(hr)
        dot = ux[:, None] * dx + uy[:, None] * dy
        foc = jnp.degrees(jnp.arccos(jnp.clip(dot / (dd + 1e-10), -1.0, 1.0)))   # [i,j] ATA of i towards j
        hd = jnp.degrees(jnp.arccos(jnp.clip((ux[:, None] * ux[None, :] + uy[:, None] * uy[None, :]) / (1.0 + 1e-10),
                                             -1.0, 1.0))) / 180.0
        return dd, foc, jnp.clip(hd, 0.0, 1.0)

    def _own(self, s):
        latr = jnp.clip(s['lat'] / self.ms, 0.0, 1.0)
        lonr = jnp.clip(s['lon'] / self.ms, 0.0, 1.0)
        spdn = jnp.clip(s['spd'] / jnp.where(s['ac1'], 900.0, 600.0), 0.0, 1.0)
        hn = jnp.clip(jnp.mod(s['hdg'], 359.0) / 359.0, 0.0, 1.0)
        return latr, lonr, spdn, hn

    @staticmethod
    def _shot(s):
        return ((s['burst'] > 0) | s['has_mis']).astype(jnp.float32)

    def _sorted_enemies(self, s, dd):
        team = jnp.asarray(self.team)
        enemy = (team[:, None] != team[None, :]) & s['alive'][None, :]
        dm = jnp.where(enemy, dd, jnp.inf)
        order = jnp.argsort(dm, axis=1, stable=True)[:, :self.K]
        n = jnp.sum(enemy, axis=1) * s['alive']
        kk = jnp.arange(self.K)[None, :]
        lst = jnp.where(kk < n[:, None], order, -1).astype(jnp.int32)
        lst_dn = jnp.take_along_axis(dd, jnp.maximum(lst, 0), axis=1) / self.dnorm
        return lst, jnp.where(lst >= 0, lst_dn, 0.0), n.astype(jnp.int32)

    def _sorted_friends(self, s, dd):
        team = jnp.asarray(self.team)
        fr = (team[:, None] == team[None, :]) & s['alive'][None, :] & ~jnp.eye(self.T, dtype=bool)
        dm = jnp.where(fr, dd, jnp.inf)
        order = jnp.argsort(dm, axis=1, stable=True)
        n = jnp.sum(fr, axis=1)
        return jnp.where(jnp.arange(self.T)[None, :] < n[:, None], order, -1)

    def _fri_block(self, s, P, fri):
        dd, foc, _ = P
        ar = jnp.arange(self.T)
        f = jnp.maximum(fri, 0)
        latr, lonr, _, _ = self._own(s)
        ok = (fri >= 0) & s['alive'][f]
        blk = jnp.stack([latr[f], lonr[f], jnp.clip(foc[ar, f] / 180, 0, 1), jnp.clip(foc[f, ar] / 180, 0, 1),
                         dd[ar, f] / self.dnorm], -1)
        return jnp.where(ok[:, None], blk, 0.0)

    def _fight_rows(self, s, P, tgt, tdn, fri, oflag):
        """fight_state_values for every unit -> [T, 26]; AC2 rows use the first 24 entries."""
        dd, foc, hd = P
        ar = jnp.arange(self.T)
        j = jnp.maximum(tgt, 0)
        latr, lonr, spdn, hn = self._own(s)
        fij, fji = jnp.clip(foc[ar, j] / 180, 0, 1), jnp.clip(foc[j, ar] / 180, 0, 1)
        can = jnp.clip(s['can_rem'] / s['can_max'], 0, 1)
        mis = jnp.clip(s['mis_rem'] / jnp.maximum(s['mis_max'], 1.0), 0, 1)
        base = jnp.stack([latr, lonr, spdn, hn, fij, 1.0 - fji, hd[ar, j], tdn, can], -1)
        opp = jnp.stack([latr[j], lonr[j], spdn[j], hn[j], hd[j, ar], fji, 1.0 - fij, tdn, oflag[j]], -1)
        fr = self._fri_block(s, P, fri)
        own_shot = self._shot(s)
        r1 = jnp.concatenate([base, mis[:, None], (s['mwait'] == 0).astype(jnp.float32)[:, None],
                              own_shot[:, None], opp, fr], -1)
        r2 = jnp.concatenate([base, (s['burst'] > 0).astype(jnp.float32)[:, None], opp, fr,
                              jnp.zeros((self.T, 2))], -1)
        row = jnp.where(s['ac1'][:, None], r1, r2)
        return jnp.where((s['alive'] & (tgt >= 0))[:, None], row, 0.0)

    def _esc_rows(self, s, P, e, edn, en, fri, oflag):
        """esc_state_values -> [T, 30]; e/edn: [T, >=2] nearest-first enemy ids / normalized dists, en: count."""
        dd, foc, hd = P
        ar = jnp.arange(self.T)
        latr, lonr, spdn, hn = self._own(s)
        can = jnp.clip(s['can_rem'] / s['can_max'], 0, 1)
        mis = jnp.clip(s['mis_rem'] / jnp.maximum(s['mis_max'], 1.0), 0, 1)
        shot = self._shot(s)
        blks = []
        for k in range(2):
            o = jnp.maximum(e[:, k], 0)
            b = jnp.stack([latr[o], lonr[o], spdn[o], hn[o], hd[o, ar], jnp.clip(foc[ar, o] / 180, 0, 1),
                           jnp.clip(foc[o, ar] / 180, 0, 1), edn[:, k], oflag[o]], -1)
            blks.append(jnp.where((en > k)[:, None], b, 0.0))
        fr = self._fri_block(s, P, fri)
        r1 = jnp.concatenate([jnp.stack([latr, lonr, spdn, hn, can, mis, shot], -1)] + blks + [fr], -1)
        r2 = jnp.concatenate([jnp.stack([latr, lonr, spdn, hn, can, shot], -1)] + blks + [fr, jnp.zeros((self.T, 1))], -1)
        row = jnp.where(s['ac1'][:, None], r1, r2)
        return jnp.where((s['alive'] & (en > 0))[:, None], row, 0.0)

    def _low_obs(self, s):
        """LowLevelEnv.state(): nearest opponent, fixed partner; also stores opp_to_attack (s['tgt'])."""
        P = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, P[0])
        flags = self._shot(s)
        tgt = jnp.where(n > 0, lst[:, 0], -1)
        if self.mode == 'fight':
            rows = self._fight_rows(s, P, tgt, lst_dn[:, 0], self.partner, flags)
        else:
            rows = self._esc_rows(s, P, lst, lst_dn, n, self.partner, flags)
        s = dict(s)
        s['tgt'] = s['tgt'].at[:self.N].set(tgt[:self.N])
        return rows[:self.N], s

    def _cmd_obs(self, s):
        """HighLevelEnv.state(): 34-dim commander obs for agents; stores the sorted enemy lists of all units."""
        dd, foc, hd = P = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, dd)
        fr = self._sorted_friends(s, dd)
        ar = jnp.arange(self.T)
        latr, lonr, spdn, hn = self._own(s)
        parts = [jnp.stack([latr, lonr, spdn, hn], -1)]
        for k in range(2):
            o = jnp.maximum(lst[:, k], 0)
            fio, foi = jnp.clip(foc[ar, o] / 180, 0, 1), jnp.clip(foc[o, ar] / 180, 0, 1)
            b = jnp.stack([latr[o], lonr[o], spdn[o], hn[o], hd[o, ar], fio, foi, 1 - fio, 1 - foi, lst_dn[:, k]], -1)
            parts.append(jnp.where((n > k)[:, None], b, 0.0))
        for k in range(2):
            parts.append(self._fri_block(s, P, fr[:, k]) if k < self.T else jnp.zeros((self.T, 5)))
        rows = jnp.concatenate(parts, -1)
        rows = jnp.where((s['alive'] & (n > 0))[:, None], rows, 0.0)
        s = dict(s)
        s.update(lst=lst, lst_dn=lst_dn, lst_n=n)
        return rows[:self.N], s

    # ------------------------------------------------------------------ actions
    def _fire_missile(self, s, try_mask, tgt):
        """Rafale.fire_missile for all units in try_mask (radar cone [h, h+120] quirk; launch in id order)."""
        j = jnp.maximum(tgt, 0)
        d_km, brg = geo_inv(s['lat'], s['lon'], s['lat'][j], s['lon'][j])
        radar = jnp.floor(jnp.abs(shd(jnp.mod(s['hdg'] + 60.0, 360.0), brg))) <= 60.0
        ok = try_mask & (tgt >= 0) & ~s['has_mis'] & (s['mis_rem'] > 0) & (d_km <= 111.0) & radar
        seq = s['seq_ctr'] + jnp.cumsum(ok.astype(jnp.int32)) - 1
        s = dict(s)
        s.update(r_alive=s['r_alive'] | ok, r_lat=jnp.where(ok, s['lat'], s['r_lat']),
                 r_lon=jnp.where(ok, s['lon'], s['r_lon']), r_hdg=jnp.where(ok, s['hdg'], s['r_hdg']),
                 r_nhdg=jnp.where(ok, s['hdg'], s['r_nhdg']), r_spd=jnp.where(ok, 500.0, s['r_spd']),
                 r_tgt=jnp.where(ok, j, s['r_tgt']), r_t0=jnp.where(ok, s['t'], s['r_t0']),
                 r_seq=jnp.where(ok, seq, s['r_seq']), seq_ctr=s['seq_ctr'] + jnp.sum(ok),
                 has_mis=s['has_mis'] | ok, mis_rem=s['mis_rem'] - ok)
        return s

    def _base_action(self, s, mask, a, tgt, key):
        """_take_base_action for every unit in `mask` (a: [T,4] ints). Returns state, escape penalties [T]."""
        maxspd = jnp.where(s['ac1'], 900.0, 600.0)
        s = dict(s)
        s['nhdg'] = jnp.where(mask, jnp.mod(s['hdg'] + (a[:, 0] - 6) * 15.0, 360.0), s['nhdg'])
        s['nspd'] = jnp.where(mask, 100.0 + ((maxspd - 100.0) / 8.0) * a[:, 1], s['nspd'])
        fire_c = mask & (a[:, 2] == 1) & (s['can_rem'] > 0)
        pen = jnp.where(fire_c & (s['can_rem'] < 90), -0.1, 0.0)
        s['burst'] = jnp.where(fire_c, jnp.minimum(s['can_rem'], jnp.where(s['ac1'], 5.0, 3.0)), s['burst'])
        try_m = mask & s['ac1'] & (a[:, 3] == 1) & (tgt >= 0) & (s['mis_rem'] > 0) & ~s['has_mis'] & (s['mwait'] == 0)
        s = self._fire_missile(s, try_m, tgt)
        wait = _randint(key, 8, 12, (self.T,)) if self.hl else _randint(key, 7, 17, (self.T,))
        s['mwait'] = jnp.where(try_m, wait, s['mwait'])           # cooldown set even if the launch failed
        pen = pen + jnp.where(try_m & (s['mis_rem'] < 3), -0.1, 0.0)
        s['mwait'] = jnp.where(mask & (s['mwait'] > 0) & ~s['has_mis'], s['mwait'] - 1, s['mwait'])
        return s, pen

    def _scripted_opps(self, s, key):
        """Level 1-3 opponents (env_hetero.__opp_level1/2/3), processed in id order with the shared L3 escape state."""
        N, T, L = self.N, self.T, self.level
        dd, foc, _ = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, dd)
        near, ndn, has = lst[:, 0], lst_dn[:, 0], n > 0
        latr, lonr, _, _ = self._own(s)
        keys = jax.random.split(key, T)
        for i in range(N, T):
            k = jax.random.split(keys[i], 12)
            alive = s['alive'][i]
            onehot = jnp.arange(T) == i
            s = dict(s)
            if L in (1, 2):
                if L == 2:
                    s['burst'] = jnp.where(onehot & alive, jnp.minimum(s['can_rem'], jnp.where(s['ac1'], 5.0, 3.0)),
                                           s['burst'])
                    turn = (s['steps'] <= 5) | (jnp.mod(s['steps'], _randint(k[0], 35, 45)) <= 5)
                    sign = jnp.where(_randint(k[1], 0, 1) == 0, 90.0, -90.0)
                    nh = jnp.mod(s['hdg'][i] + sign, 360.0)
                    ns = 100.0 + _randint(k[2], 0, 4) * 75.0
                    s['nhdg'] = jnp.where(onehot & alive & turn, nh, s['nhdg'])
                    s['nspd'] = jnp.where(onehot & alive & turn, ns, s['nspd'])
                cond = (alive & ~s['has_mis'][i] & (jnp.mod(s['steps'], 40) < 3) & (_randint(k[3], 0, 1) == 1)
                        & (s['mwait'][i] == 0) & s['ac1'][i] & has[i])
                s = self._fire_missile(s, onehot & cond, jnp.where(onehot, near, -1))
                s['mwait'] = jnp.where(onehot & cond, 5, s['mwait'])
            else:  # level 3
                roll = alive & (jnp.mod(s['steps'], 60) == 0) & ~s['esc_flag']
                coin = _randint(k[0], 0, 1) == 1
                flag = jnp.where(roll, coin, s['esc_flag'])
                timer = jnp.where(roll & coin, jnp.floor(_uni(k[1], 20.0, 30.0)).astype(jnp.int32), s['esc_timer'])
                # escaping opponent: diagonal flight back to the map centre
                y, x = latr[i], lonr[i]
                lo = jnp.where(y < 0.5, jnp.where(x < 0.5, 30.0, 300.0), jnp.where(x < 0.5, 120.0, 210.0))
                h_esc = jnp.floor(_uni(k[2], lo, lo + 30.0))
                sp_esc = jnp.floor(_uni(k[3], 300.0, 600.0))
                fire_esc = _randint(k[4], 0, 1) == 1
                # hard-coded pursuit of the nearest agent
                ag = jnp.maximum(near[i], 0)
                hdg = s['hdg'][i]
                x0, y0 = s['lon'][i], s['lat'][i]
                x1 = x0 + jnp.round(jnp.sin(jnp.mod(hdg, 360.0) * D2R), 3)
                y1 = y0 + jnp.round(jnp.cos(jnp.mod(hdg, 360.0) * D2R), 3)
                val = (x1 - x0) * (s['lat'][ag] - y0) - (s['lon'][ag] - x0) * (y1 - y0)
                sign = jnp.where(val < 0, 1.0, -1.0)
                rr = _uni(k[5], 0.7, 1.3)
                fo = foc[i, ag]
                d = ndn[i]
                sp_h = jnp.floor(_uni(k[6], 100.0, 400.0))
                h_h = jnp.where(has[i] & (d > 0.008) & (fo > 4), jnp.mod(hdg + rr * sign * fo, 360.0), hdg)
                sp_far = jnp.where(fo < 30, jnp.floor(_uni(k[7], 500.0, 800.0)), jnp.floor(_uni(k[8], 100.0, 500.0)))
                sp_h = jnp.where(has[i] & (d > 0.05), sp_far, sp_h)
                sp_h = jnp.where(s['ac1'][i], sp_h, jnp.minimum(sp_h, 600.0))
                fire_h = has[i] & (d < 0.03) & (fo < 10)
                firem_h = has[i] & (d < 0.09) & (fo < 5)
                esc = flag
                heading = jnp.where(esc, h_esc, h_h)
                speed = jnp.where(esc, sp_esc, sp_h)
                fire = jnp.where(esc, fire_esc, fire_h)
                firem = jnp.where(esc, False, firem_h)
                timer = jnp.where(esc, timer - 1, timer)
                flag = jnp.where(esc & (timer <= 0), False, flag)
                s['esc_flag'] = jnp.where(alive, flag, s['esc_flag'])
                s['esc_timer'] = jnp.where(alive, timer, s['esc_timer'])
                m = onehot & alive
                s['nhdg'] = jnp.where(m, heading, s['nhdg'])
                s['nspd'] = jnp.where(m, speed, s['nspd'])
                s['burst'] = jnp.where(m & fire, jnp.minimum(s['can_rem'], jnp.where(s['ac1'], 5.0, 3.0)), s['burst'])
                cond = alive & firem & ~s['has_mis'][i] & (s['mwait'][i] == 0) & s['ac1'][i]
                s = self._fire_missile(s, onehot & cond, jnp.where(onehot, near, -1))
                s['mwait'] = jnp.where(onehot & cond, 10, s['mwait'])
        return s

    def _fsp_opps(self, s, key, opp_params):
        """Level 4/5 opponents: frozen policies (argmax). They observe the agents' current-step shot flags."""
        P = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, P[0])
        flags = self._shot(s)
        tgt = jnp.where(n > 0, lst[:, 0], -1)
        obs_f = self._fight_rows(s, P, tgt, lst_dn[:, 0], self.partner, flags)
        obs_e = self._esc_rows(s, P, lst, lst_dn, n, self.partner, flags)
        acts = [frozen_policy_actions(kind, p, obs_f if kind == 'fight' else obs_e, s['ac1'])
                for kind, p in zip(self.opp_pool, opp_params)]
        a = acts[0]
        for k in range(1, len(acts)):
            a = jnp.where(s['opp_k'] == k, acts[k], a)
        s = dict(s)
        s['tgt'] = s['tgt'].at[self.N:].set(tgt[self.N:])
        mask = s['alive'] & ~self.is_ag
        s, _ = self._base_action(s, mask, a, s['tgt'], key)
        return s

    # ------------------------------------------------------------------ simulator tick
    def _tick(self, s, key):
        T = self.T
        team = jnp.asarray(self.team)
        alive0, ralive0 = s['alive'], s['r_alive']
        ka, kb = jax.random.split(key)
        u_hit = jax.random.uniform(ka, (T, T))
        u_mis = _uni(kb, 0.95, 1.05, (T,))
        killer = jnp.full((T,), -1, jnp.int32)
        kind = jnp.zeros((T,), jnp.int32)          # 0 cannon, 1 rocket
        ar = jnp.arange(T)

        def ac_body(i, c):
            s, killer, kind = c
            upd = alive0[i]
            ac1 = s['ac1'][i]
            h, nh = s['hdg'][i], s['nhdg'][i]
            maxd = jnp.where(ac1, 5.0, 3.5)
            dl = shd(h, nh)
            h2 = jnp.where(jnp.abs(dl) <= maxd, nh, jnp.mod(h + jnp.where(dl >= 0, maxd, -maxd), 360.0))
            h2 = jnp.where(h != nh, h2, h)
            sp, ns = s['spd'][i], s['nspd'][i]
            maxa = jnp.where(ac1, 35.0, 28.0)
            ds = ns - sp
            sp2 = jnp.where(jnp.abs(ds) <= maxa, ns, sp + jnp.where(ds >= 0, maxa, -maxa))
            sp2 = jnp.where(sp != ns, sp2, sp)
            # cannon
            burst, rem = s['burst'][i], s['can_rem'][i]
            firing = upd & (burst > 0)
            d_km, brg = geo_inv(s['lat'][i], s['lon'][i], s['lat'], s['lon'])
            cone = (d_km < jnp.where(ac1, 2.0, 4.5)) & (jnp.abs(shd(h2, brg)) <= jnp.where(ac1, 5.0, 3.5))
            ff = (team != team[i]) | FRIENDLY_KILL
            p = jnp.where(ac1, self.p_tick[0], self.p_tick[1])
            hit = firing & s['alive'] & (ar != i) & cone & ff & (u_hit[i] < p)
            killer = jnp.where(hit, i, killer)
            kind = jnp.where(hit, 0, kind)
            # missile steering (unguided: multiplicative heading noise)
            hm, rok = s['has_mis'][i], s['r_alive'][i]
            steer = upd & hm & rok
            r_nh = jnp.clip(s['r_hdg'][i] * u_mis[i], 0.0, 359.0)
            # move
            la2, lo2 = geo_move(s['lat'][i], s['lon'][i], h2, sp2 * KN2MS)
            mv = upd & (sp2 > 0)
            s = dict(s)
            s['alive'] = s['alive'] & ~hit
            s['hdg'] = s['hdg'].at[i].set(jnp.where(upd, h2, h))
            s['spd'] = s['spd'].at[i].set(jnp.where(upd, sp2, sp))
            s['burst'] = s['burst'].at[i].set(jnp.where(firing, jnp.maximum(burst - 1.0, 0.0), burst))
            s['can_rem'] = s['can_rem'].at[i].set(jnp.where(firing, jnp.maximum(rem - 1.0, 0.0), rem))
            s['has_mis'] = s['has_mis'].at[i].set(jnp.where(upd & hm & ~rok, False, hm))
            s['r_nhdg'] = s['r_nhdg'].at[i].set(jnp.where(steer, r_nh, s['r_nhdg'][i]))
            s['lat'] = s['lat'].at[i].set(jnp.where(mv, la2, s['lat'][i]))
            s['lon'] = s['lon'].at[i].set(jnp.where(mv, lo2, s['lon'][i]))
            return s, killer, kind

        s, killer, kind = jax.lax.fori_loop(0, T, ac_body, (s, killer, kind))
        order = jnp.argsort(jnp.where(ralive0, s['r_seq'], jnp.iinfo(jnp.int32).max))

        def r_body(k, c):
            s, killer, kind = c
            j = order[k]
            upd = ralive0[j]
            tg = s['r_tgt'][j]
            d_t, _ = geo_inv(s['r_lat'][j], s['r_lon'][j], s['lat'][tg], s['lon'][tg])
            hit_t = upd & (d_t < 1.0) & s['alive'][tg]
            fid = jnp.where(j == 1, 0, 1)          # proximity fuse hard-coded to aircraft id 2 (bug)
            d_f, _ = geo_inv(s['r_lat'][j], s['r_lon'][j], s['lat'][fid], s['lon'][fid])
            hit_f = upd & ~hit_t & FRIENDLY_KILL & s['alive'][fid] & (d_f < 1.0)
            life = s['t'] - s['r_t0'][j]
            eol = upd & ~hit_t & ~hit_f & (life > 10)
            cont = upd & ~hit_t & ~hit_f & ~eol
            s = dict(s)
            alive = s['alive'].at[tg].set(jnp.where(hit_t, False, s['alive'][tg]))
            s['alive'] = alive.at[fid].set(jnp.where(hit_f, False, alive[fid]))
            killer = killer.at[tg].set(jnp.where(hit_t, j, killer[tg]))
            kind = kind.at[tg].set(jnp.where(hit_t, 1, kind[tg]))
            killer = killer.at[fid].set(jnp.where(hit_f, j, killer[fid]))
            kind = kind.at[fid].set(jnp.where(hit_f, 1, kind[fid]))
            rh, rnh = s['r_hdg'][j], s['r_nhdg'][j]
            dl = shd(rh, rnh)
            rh2 = jnp.where(rh != rnh, jnp.where(jnp.abs(dl) <= 10.0, rnh, rh + jnp.where(dl >= 0, 10.0, -10.0)), rh)
            rs = jnp.asarray(ROCKET_SPEED)[jnp.clip(life, 0, 10)]
            la2, lo2 = geo_move(s['r_lat'][j], s['r_lon'][j], rh2, rs * KN2MS)
            s['r_alive'] = s['r_alive'].at[j].set(jnp.where(hit_t | hit_f | eol, False, s['r_alive'][j]))
            s['r_hdg'] = s['r_hdg'].at[j].set(jnp.where(cont, rh2, rh))
            s['r_spd'] = s['r_spd'].at[j].set(jnp.where(cont, rs, s['r_spd'][j]))
            s['r_lat'] = s['r_lat'].at[j].set(jnp.where(cont, la2, s['r_lat'][j]))
            s['r_lon'] = s['r_lon'].at[j].set(jnp.where(cont, lo2, s['r_lon'][j]))
            return s, killer, kind

        s, killer, kind = jax.lax.fori_loop(0, T, r_body, (s, killer, kind))
        s['t'] = s['t'] + 1
        return s, killer, kind

    # ------------------------------------------------------------------ rewards & bookkeeping
    def _combat(self, s, killer, kind):
        """Out-of-boundary removal + per-agent combat outcome terms. Returns state and outcome dict."""
        N = self.N
        inb = (s['lat'] >= 0) & (s['lat'] <= self.ms) & (s['lon'] >= 0) & (s['lon'] <= self.ms)
        oob = s['alive'] & ~inb
        s = dict(s)
        s['alive'] = s['alive'] & ~oob
        is_ag = self.is_ag
        by_ag = (killer >= 0) & (killer < N)
        by_op = killer >= N
        ka = killer[None, :] == jnp.arange(N)[:, None]              # [agent, victim]
        kills_opp = ka & ~is_ag[None, :]
        kills_fr = ka & is_ag[None, :]
        t1 = s['ac1'][:N].astype(jnp.float32)
        tw = jnp.stack([t1, 1 - t1], -1)                              # agent type one-hot [N,2]
        dead_ag = (by_ag | by_op)[:N] | oob[:N]
        s['st_k'] = s['st_k'] + jnp.sum(jnp.sum(kills_opp, 1)[:, None] * tw, 0)
        s['st_fk'] = s['st_fk'] + jnp.sum(jnp.sum(kills_fr, 1)[:, None] * tw, 0)
        s['st_d'] = s['st_d'] + jnp.sum(dead_ag[:, None] * tw, 0)
        s['st_ag_by_opp'] = s['st_ag_by_opp'] + jnp.sum(by_op[:N])
        s['st_opp_by_ag'] = s['st_opp_by_ag'] + jnp.sum(by_ag[N:])
        s['st_oob'] = s['st_oob'] + jnp.sum(oob[:N])
        out = dict(oob=oob[:N], kills_opp=kills_opp, kills_fr=kills_fr, by_op=by_op[:N], by_ag=by_ag[:N],
                   event=jnp.any(oob) | jnp.any(killer >= 0))
        return s, out

    def _low_rewards(self, s, killer, kind, ostat):
        N = self.N
        s, o = self._combat(s, killer, kind)
        if self.mode == 'fight':
            rk = 1.0 + 0.5 * s['mis_rem'][:N] / jnp.maximum(s['mis_max'][:N], 1.0)
            ck = (0.5 + 0.5 * s['can_rem'][:N] / s['can_max'][:N]) + (0.5 + 0.5 * ostat)
            per = jnp.where(kind[None, :] == 1, rk[:, None], ck[:, None])
            r = jnp.sum(o['kills_opp'] * per, 1)
        else:
            r = jnp.zeros((N,))
        r = r - 2.0 * jnp.sum(o['kills_fr'], 1) - 2.0 * o['by_op'] - 5.0 * o['oob']
        if FRIENDLY_PUNISH:
            r = r - 2.0 * o['by_ag']
        if self.mode == 'escape' and ESC_DIST_REW:
            dd = self._planar(s)[0]
            lst, _, n = self._sorted_enemies(s, dd)
            for k in range(self.K):
                o_k = jnp.maximum(lst[:N, k], 0)
                dk = dd[jnp.arange(N), o_k]
                valid = (n[:N] > k) & s['alive'][:N]
                spd = s['spd'][:N]
                near = (dk < 0.06) * (-0.02 / (k + 1)) * (1.0 + (spd < 200))
                far = (dk > 0.13) * (0.02 / (k + 1)) * (1.0 + (spd > 500))
                r = r + jnp.where(valid, near + far, 0.0)
        incl = s['alive'][:N] | o['oob'] | o['by_op'] | (FRIENDLY_PUNISH & o['by_ag'])
        return s, jnp.where(incl, r, 0.0), o['event']

    def _hl_rewards(self, s, killer, kind):
        N = self.N
        s, o = self._combat(s, killer, kind)
        r = 1.0 * jnp.sum(o['kills_opp'], 1) - 1.0 * o['by_op'] + self.oob_pen * o['oob']
        incl = s['alive'][:N] | o['oob'] | o['by_op']
        return s, jnp.where(incl, r, 0.0), o['event']

    def _done(self, s):
        na = jnp.sum(s['alive'][:self.N])
        no = jnp.sum(s['alive'][self.N:])
        done = (na == 0) | (no == 0) | (s['steps'] >= self.horizon)
        info = dict(win=(no == 0) & (s['steps'] < self.horizon), loss=(na == 0) & (s['steps'] < self.horizon),
                    draw=(s['steps'] >= self.horizon) & (na > 0) & (no > 0))
        return done, info

    # ------------------------------------------------------------------ step: low level
    def step(self, s, act, key, opp_params=()):
        if self.hl:
            return self.hl_step(s, act, key, opp_params)
        N, T = self.N, self.T
        ks = jax.random.split(key, 3)
        s = dict(s)
        s['steps'] = s['steps'] + 1
        alive_act = s['alive']
        a = jnp.zeros((T, 4), jnp.int32).at[:N].set(act.astype(jnp.int32))
        dd, foc, _ = self._planar(s)
        tg = jnp.maximum(s['tgt'][:N], 0)
        ostat = jnp.where(s['tgt'][:N] >= 0, jnp.clip(foc[tg, jnp.arange(N)] / 180.0, 0, 1), 0.0)  # opp_stats
        s, pen = self._base_action(s, alive_act & self.is_ag, a, s['tgt'], ks[0])
        pen = pen[:N] if self.mode == 'escape' else jnp.zeros((N,))
        if self.level >= 4:
            s = self._fsp_opps(s, ks[1], opp_params)
        else:
            s = self._scripted_opps(s, ks[1])
        s, killer, kind = self._tick(s, ks[2])
        s, r, _ = self._low_rewards(s, killer, kind, ostat)
        rew = jnp.where(alive_act[:N], pen + r, 0.0)
        obs, s = self._low_obs(s)
        done, info = self._done(s)
        trace = self._trace(s) if self.record else None
        return s, obs, rew, done, info, trace

    def _trace(self, s):
        return {k: s[k] for k in ('lat', 'lon', 'hdg', 'alive', 'r_lat', 'r_lon', 'r_alive', 'burst')}

    # ------------------------------------------------------------------ step: commander level
    def hl_step(self, s, cmd, key, llp):
        """One commander step. cmd: [N] commander actions (0 escape, k attack k-th nearest opponent).
        llp: dict(fight_ag=params, fight_opp=params, escape=params) of frozen low-level policies."""
        N, T = self.N, self.T
        ks = jax.random.split(key, 6)
        dd, foc, _ = self._planar(s)
        ar = jnp.arange(N)
        lst, n = s['lst'], s['lst_n']
        a = cmd.astype(jnp.int32)
        inval = (a > 0) & (a > n[:N])
        ca_ag = jnp.where(inval, 1, a)
        tg = jnp.maximum(lst[ar, jnp.maximum(ca_ag - 1, 0)], 0)
        cl = jnp.maximum(lst[ar, 0], 0)
        r_f = jnp.where((dd[ar, tg] < 0.1) & (foc[ar, tg] < 15) & (foc[tg, ar] > 40), 0.1, 0.0)
        r_e = jnp.where((dd[cl, ar] < 0.1) & (foc[cl, ar] < 15) & (foc[ar, cl] > 40), 0.1, 0.0)
        r0 = jnp.where(a > 0, r_f, r_e) if HIER_ACTION_ASSESS else jnp.zeros((N,))
        r0 = jnp.where(inval, -0.1, r0)
        rew = jnp.where(s['alive'][:N], r0, 0.0)
        # opponents: fight with prob ratio; attack a random non-nearest agent with prob 0.75
        no = n[N:]
        fight = jax.random.uniform(ks[0], (self.M,)) < HIER_OPP_FIGHT_RATIO / 100.0
        other = (no > 1) & (jax.random.uniform(ks[1], (self.M,)) < 0.75)
        pick = jax.random.randint(ks[2], (self.M,), 2, jnp.maximum(no, 2) + 1)
        ca_op = jnp.where(fight, jnp.where(other, pick, 1), 0)
        ca = jnp.concatenate([ca_ag, ca_op])
        s = dict(s)
        s['ca'] = jnp.where(s['alive'], ca, -1)

        def sub(c, idx):
            s, rew, active, key = c
            key, k1 = jax.random.split(key)
            s2, r_t, event = self._hl_substep(s, k1, llp)
            situation = (idx > 10) & self._surrounding(s2)
            s2['steps'] = s2['steps'] + 1
            s = tree_where(active, s2, s)
            rew = jnp.where(active, rew + r_t, rew)
            tr = (self._trace(s), active) if self.record else None
            return (s, rew, active & ~event & ~situation, key), tr

        (s, rew, _, _), trace = jax.lax.scan(sub, (s, rew, jnp.array(True), ks[3]), jnp.arange(16))
        obs, s = self._cmd_obs(s)
        done, info = self._done(s)
        return s, obs, rew, done, info, trace

    def _surrounding(self, s):
        dd, foc, _ = self._planar(s)
        N = self.N
        pair = s['alive'][:N, None] & s['alive'][None, N:]
        ev = (dd[:N, N:] < 0.1) & ((foc[:N, N:] < 15) | (foc[N:, :N].T < 15))
        return jnp.any(pair & ev)

    def _hl_substep(self, s, key, llp):
        N, T = self.N, self.T
        ks = jax.random.split(key, 3)
        P = self._planar(s)
        ar = jnp.arange(T)
        ca, lst, lst_dn, n = s['ca'], s['lst'], s['lst_dn'], s['lst_n']
        ci = jnp.maximum(ca - 1, 0)
        tgt_f = jnp.where(ca >= 1, lst[ar, ci], -1)
        tdn_f = lst_dn[ar, ci]
        far = lst[ar, jnp.maximum(n - 1, 0)]                   # escape: missile at the farthest (list[-1]) quirk
        mtgt = jnp.where(ca >= 1, tgt_f, jnp.where(ca == 0, far, -1))
        fr = self._sorted_friends(s, P[0])[:, 0]              # nearest living teammate, recomputed each tick
        use_esc = ca == 0

        def acts(s, rows, fparams):
            flags = self._shot(s)
            of = self._fight_rows(s, P, tgt_f, tdn_f, fr, flags)[rows]
            oe = self._esc_rows(s, P, lst, lst_dn, n, fr, flags)[rows]
            return frozen_ll_actions(fparams, llp['escape'], of, oe, s['ac1'][rows], use_esc[rows])

        a_ag = acts(s, slice(0, N), llp['fight_ag'])
        a = jnp.zeros((T, 4), jnp.int32).at[:N].set(a_ag)
        s, _ = self._base_action(s, s['alive'] & self.is_ag, a, mtgt, ks[0])
        a_op = acts(s, slice(N, T), llp['fight_opp'])          # opponents see agents' current shot flags
        a = jnp.zeros((T, 4), jnp.int32).at[N:].set(a_op)
        s, _ = self._base_action(s, s['alive'] & ~self.is_ag, a, mtgt, ks[1])
        s, killer, kind = self._tick(s, ks[2])
        return self._hl_rewards(s, killer, kind)


def make_batched(env):
    """jit+vmap reset/step for a batch of envs; step auto-resets finished envs."""
    reset_b = jax.vmap(env.reset)

    def step_b(s, act, keys, params):
        return jax.vmap(env.step, in_axes=(0, 0, 0, None))(s, act, keys, params)
    return reset_b, step_b

Quick sanity check of the simulator: random agent actions vs the scripted L3 opponents (no networks needed).

In [ ]:
def _env_sanity(n_envs=64, steps=300):
    env = Env('low', level=3, mode='fight')
    reset_b, step_b = make_batched(env)
    key = jax.random.PRNGKey(0)
    s, obs = jax.jit(reset_b)(jax.random.split(key, n_envs))

    @jax.jit
    def run(s, key):
        def body(c, k):
            s, ret, dn = c
            k1, k2 = jax.random.split(k)
            a = jax.random.randint(k1, (n_envs, 2, 4), 0, jnp.array([13, 9, 2, 2]))
            s2, obs, r, d, info, _ = step_b(s, a, jax.random.split(k2, n_envs), ())
            live = ~dn
            return (tree_where(live, s2, s), ret + jnp.where(live[:, None], r, 0), dn | d), (obs, info['win'] & live & d)
        (s, ret, dn), (obs, wins) = jax.lax.scan(body, (s, jnp.zeros((n_envs, 2)), jnp.zeros(n_envs, bool)),
                                                 jax.random.split(key, steps))
        return s, ret, dn, obs, wins
    t0 = time.time()
    s, ret, dn, obs, wins = jax.block_until_ready(run(s, key))
    print(f'L3 random-vs-scripted: {n_envs} envs x {steps} steps in {time.time() - t0:.1f}s (incl. compile); '
          f'done={int(dn.sum())}, agent wins={int(wins.sum())}, mean team return={float(ret.sum(1).mean()):.2f}, '
          f'obs range=[{float(obs.min()):.2f}, {float(obs.max()):.2f}], rockets fired={int(s["seq_ctr"].sum())}')

_env_sanity(8 if SMOKE else 256)

## 4. Networks (models/) and PPO (RLlib 2.4 semantics)

**Networks.** All are pure-JAX parameter dicts. Initialization follows the original:
* `SlimFC` layers: orthogonal weights, zero bias;
* multi-head attention: xavier `in_proj` and torch-default `out_proj`, zero biases;
* GRU: torch default U(±1/√H).

The model families:

| family | original | actor | critic (centralized) |
|---|---|---|---|
| `fight_sa` | `Fight1/Fight2` | FC200(own) ‖ FC200(rest) ‖ L2norm(FC100 + MHA(100,2 heads)) → **SHARED 500** → logits | FC175(own obs+act) ‖ FC175(partner obs+act) ‖ L2norm(FC150 + MHA(150)) → SHARED → V |
| `escape` | `Esc1/Esc2` | FC150 ‖ FC250 ‖ FC100 → SHARED → logits | FC500(all obs+acts) → SHARED → V |
| `fight_fc` | Fig. 6 "FC" baseline | 2×FC500 tanh → logits | 2×FC500 tanh on all obs+acts → V |
| `cmd_gru` | `CommanderGru` | FC50 ‖ FC200 ‖ FC50 ‖ L2norm(FC200 + GRU200) → SHARED → 3 logits | 3×FC100 ‖ L2norm(FC200 + GRU200) → SHARED → V |
| `cmd_sa`, `cmd_fc` | Fig. 10 baselines | GRU replaced by MHA(200) / 2×FC500 | same replacement |

**Shared layer.** `SHARED` (500→500 tanh) is one parameter shared by AC1 and AC2 and by the actor and the critic, as in
the module-level global `SHARED_LAYER`. Each low-level policy has its own Adam state over {its weights, SHARED}, and the
two policies are updated one after the other (ac1 then ac2), as in RLlib's `simple_optimizer` path.

**PPO.** Matches RLlib 2.4 `PPOTorchPolicy`:
* clipped surrogate, plus the squared value error clipped at `vf_clip`, plus `kl_coeff`·KL(old‖new);
* the KL coefficient adapts once per iteration from the last epoch's mean KL;
* advantages are standardized per policy;
* GAE with the horizon treated as terminal;
* dead agents keep producing zero-obs rows.

The rollout is a `lax.scan` over vmapped envs, and rollout + GAE + all SGD epochs are one jit-compiled function.

**RLLIB_FAITHFUL** quirks:
* **SA nets:** at training time, attention runs over each 20-step chunk of an episode **without a causal mask**
  (it also attends to future steps). RLlib's zero-padding to 20 rows is emulated exactly:
  * padded keys enter the softmax with multiplicity n_pad (a log-count bias);
  * padded rows enter the unmasked loss mean, with advantage 0, value target 0 and KL to the uniform distribution.

  At rollout time each step is a length-1 sequence.
* **Critic:** rollout values (used for GAE) are computed with zero action inputs. The critic is trained with the real
  (normalized) actions.
* **GRU commander:** the loss is masked (no padding). Chunk-start hidden states come from the rollout, as in RLlib.

In [ ]:
def _orth(key, din, dout):
    return jax.nn.initializers.orthogonal()(key, (din, dout), jnp.float32)


def dense_init(key, din, dout):
    return {'w': _orth(key, din, dout), 'b': jnp.zeros((dout,), jnp.float32)}


def dense(p, x):
    return x @ p['w'] + p['b']


def mha_init(key, E):
    k1, k2 = jax.random.split(key)
    b = math.sqrt(6.0 / (E + 3 * E))
    w = jax.random.uniform(k1, (E, 3 * E), minval=-b, maxval=b)
    wo = jax.random.uniform(k2, (E, E), minval=-1 / math.sqrt(E), maxval=1 / math.sqrt(E))
    z = jnp.zeros((E,))
    return dict(wq=w[:, :E], wk=w[:, E:2 * E], wv=w[:, 2 * E:], bq=z, bk=z, bv=z, wo=wo, bo=z)


def gru_init(key, din, H):
    ks = jax.random.split(key, 4)
    u = lambda k, s: jax.random.uniform(k, s, minval=-1 / math.sqrt(H), maxval=1 / math.sqrt(H))
    return dict(wi=u(ks[0], (din, 3 * H)), wh=u(ks[1], (H, 3 * H)), bi=u(ks[2], (3 * H,)), bh=u(ks[3], (3 * H,)))


def gru_cell(p, x, h):
    gi, gh = x @ p['wi'] + p['bi'], h @ p['wh'] + p['bh']
    H = h.shape[-1]
    r = jax.nn.sigmoid(gi[..., :H] + gh[..., :H])
    z = jax.nn.sigmoid(gi[..., H:2 * H] + gh[..., H:2 * H])
    n = jnp.tanh(gi[..., 2 * H:] + r * gh[..., 2 * H:])
    return (1 - z) * n + z * h


def l2n(x):
    return x / jnp.maximum(jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-12)


def mha_single(p, x):
    """nn.MultiheadAttention on a length-1 sequence (rollout / FSP inference)."""
    return (x @ p['wv'] + p['bv']) @ p['wo'] + p['bo']


def mha_window(p, x, allowed, n_pad, x_pad, heads=2):
    """Non-causal MHA over one 20-step window x [W,E]. allowed[t,u]: same episode chunk.
    If x_pad is given, RLlib's zero-padded rows are emulated: each row's chunk has n_pad[t] pad keys (identical, so
    they enter the softmax with a log(n_pad) bias), and the output of a pad *query* row is also returned."""
    W, E = x.shape
    dh = E // heads
    proj = lambda y, w, b: (y @ p[w] + p[b])
    q = proj(x, 'wq', 'bq').reshape(W, heads, dh)
    k = proj(x, 'wk', 'bk').reshape(W, heads, dh)
    v = proj(x, 'wv', 'bv').reshape(W, heads, dh)
    sc = 1.0 / math.sqrt(dh)
    lg = jnp.einsum('qhd,khd->hqk', q, k) * sc
    lg = jnp.where(allowed[None], lg, -1e30)
    if x_pad is None:
        w = jax.nn.softmax(lg, -1)
        return (jnp.einsum('hqk,khd->qhd', w, v).reshape(W, E) @ p['wo'] + p['bo']), None
    kp = proj(x_pad, 'wk', 'bk').reshape(heads, dh)
    vp = proj(x_pad, 'wv', 'bv').reshape(heads, dh)
    qp = proj(x_pad, 'wq', 'bq').reshape(heads, dh)
    logn = jnp.where(n_pad > 0, jnp.log(jnp.maximum(n_pad, 1.0)), -1e30)

    def attend(lg_keys, lg_pad):                   # lg_keys [h,q,W], lg_pad [h,q]
        full = jnp.concatenate([lg_keys, lg_pad[..., None]], -1)
        w = jax.nn.softmax(full, -1)
        o = jnp.einsum('hqk,khd->qhd', w[..., :W], v) + jnp.einsum('hq,hd->qhd', w[..., W], vp)
        return o.reshape(-1, E) @ p['wo'] + p['bo']
    out = attend(lg, jnp.einsum('qhd,hd->hq', q, kp) * sc + logn[None])
    lgp = jnp.einsum('hd,khd->hk', qp, k) * sc                         # pad query vs real keys
    lgp = jnp.where(allowed[None], lgp[:, None, :], -1e30)              # row t: keys of t's chunk
    lpp = jnp.broadcast_to((jnp.sum(qp * kp, -1) * sc)[:, None], (heads, W)) + logn[None]
    out_pad = attend(lgp, lpp)
    return out, out_pad


# ------------------------------------------------------------------ action distributions (MultiCategorical)
def _splits(logits, splits):
    out, o = [], 0
    for n in splits:
        out.append(logits[..., o:o + n]); o += n
    return out


def md_sample(key, logits, splits):
    ks = jax.random.split(key, len(splits))
    a = [jax.random.categorical(k, l) for k, l in zip(ks, _splits(logits, splits))]
    a = jnp.stack(a, -1)
    return jnp.pad(a, [(0, 0)] * (a.ndim - 1) + [(0, 4 - len(splits))]) if len(splits) < 4 else a


def md_argmax(logits, splits):
    a = jnp.stack([jnp.argmax(l, -1) for l in _splits(logits, splits)], -1)
    return jnp.pad(a, [(0, 0)] * (a.ndim - 1) + [(0, 4 - len(splits))]) if len(splits) < 4 else a


def md_logp(logits, a, splits):
    return sum(jnp.take_along_axis(jax.nn.log_softmax(l), a[..., i:i + 1].astype(jnp.int32), -1)[..., 0]
               for i, l in enumerate(_splits(logits, splits)))


def md_entropy(logits, splits):
    return sum(-jnp.sum(jax.nn.softmax(l) * jax.nn.log_softmax(l), -1) for l in _splits(logits, splits))


def md_kl(old, new, splits):
    tot = 0.0
    for lo, ln in zip(_splits(old, splits), _splits(new, splits)):
        po, lpo, lpn = jax.nn.softmax(lo), jax.nn.log_softmax(lo), jax.nn.log_softmax(ln)
        tot = tot + jnp.sum(po * (lpo - lpn), -1)
    return tot


ACT_NORM = jnp.array([12.0, 8.0, 1.0, 1.0])

# ------------------------------------------------------------------ low-level networks
LOW_DIMS = {  # w -> (obs dim fight, obs dim esc, act dim, logits, self-state split fight, esc splits)
    1: dict(Df=26, De=30, A=4, L=26, ss=12, es=(7, 18, 5), S=SPLITS1),
    2: dict(Df=24, De=29, A=3, L=24, ss=10, es=(6, 18, 5), S=SPLITS2)}


def low_init(family, key):
    ks = jax.random.split(key, 3)
    p = {'shared': dense_init(ks[0], 500, 500) if family != 'fight_fc' else {}}
    for w in (1, 2):
        d, o = LOW_DIMS[w], LOW_DIMS[3 - w]
        k = jax.random.split(ks[w], 12)
        if family == 'fight_sa':
            D, Dp = d['Df'], o['Df']
            p[f'ac{w}'] = dict(inp1=dense_init(k[0], d['ss'], 200), inp2=dense_init(k[1], D - d['ss'], 200),
                               inp3=dense_init(k[2], D, 100), att=mha_init(k[3], 100),
                               act_out=dense_init(k[4], 500, d['L']),
                               v1=dense_init(k[5], D + d['A'], 175), v2=dense_init(k[6], Dp + o['A'], 175),
                               v3=dense_init(k[7], D + d['A'] + Dp + o['A'], 150), att_val=mha_init(k[8], 150),
                               val_out=dense_init(k[9], 500, 1))
        elif family == 'escape':
            e = d['es']
            p[f'ac{w}'] = dict(inp1=dense_init(k[0], e[0], 150), inp2=dense_init(k[1], e[1], 250),
                               inp3=dense_init(k[2], e[2], 100), act_out=dense_init(k[3], 500, d['L']),
                               inp1_val=dense_init(k[4], 66, 500), val_out=dense_init(k[5], 500, 1))
        elif family == 'fight_fc':
            D, Dp = d['Df'], o['Df']
            p[f'ac{w}'] = dict(a1=dense_init(k[0], D, 500), a2=dense_init(k[1], 500, 500),
                               act_out=dense_init(k[2], 500, d['L']),
                               c1=dense_init(k[3], D + d['A'] + Dp + o['A'], 500), c2=dense_init(k[4], 500, 500),
                               val_out=dense_init(k[5], 500, 1))
        else:
            raise ValueError(family)
    return p


def _sh(p, x):
    return jnp.tanh(dense(p['shared'], x))


def low_actor_infer(family, p, w, obs):
    """Rollout / FSP actor (length-1 sequences). obs: [..., 26|30] padded layout."""
    q, d = p[f'ac{w}'], LOW_DIMS[w]
    if family == 'fight_sa':
        D, ss = d['Df'], d['ss']
        a, b = jnp.tanh(dense(q['inp1'], obs[..., :ss])), jnp.tanh(dense(q['inp2'], obs[..., ss:D]))
        f = jnp.tanh(dense(q['inp3'], obs[..., :D]))
        f = l2n(f + mha_single(q['att'], f))
        return dense(q['act_out'], _sh(p, jnp.concatenate([a, b, f], -1)))
    if family == 'escape':
        e = d['es']
        x = jnp.concatenate([jnp.tanh(dense(q['inp1'], obs[..., :e[0]])),
                             jnp.tanh(dense(q['inp2'], obs[..., e[0]:e[0] + e[1]])),
                             jnp.tanh(dense(q['inp3'], obs[..., e[0] + e[1]:sum(e)]))], -1)
        return dense(q['act_out'], _sh(p, x))
    D = d['Df']
    return dense(q['act_out'], jnp.tanh(dense(q['a2'], jnp.tanh(dense(q['a1'], obs[..., :D])))))


def _critic_parts(family, w, own, own_a, oth, oth_a):
    d, o = LOW_DIMS[w], LOW_DIMS[3 - w]
    D, Dp = (d['Df'], o['Df']) if family != 'escape' else (d['De'], o['De'])
    c1 = jnp.concatenate([own[..., :D], own_a[..., :d['A']]], -1)
    c2 = jnp.concatenate([oth[..., :Dp], oth_a[..., :o['A']]], -1)
    return c1, c2


def low_critic_infer(family, p, w, own, own_a, oth, oth_a):
    q = p[f'ac{w}']
    c1, c2 = _critic_parts(family, w, own, own_a, oth, oth_a)
    if family == 'fight_sa':
        y = jnp.tanh(dense(q['v3'], jnp.concatenate([c1, c2], -1)))
        y = l2n(y + mha_single(q['att_val'], y))
        z = jnp.concatenate([jnp.tanh(dense(q['v1'], c1)), jnp.tanh(dense(q['v2'], c2)), y], -1)
        return dense(q['val_out'], _sh(p, z))[..., 0]
    if family == 'escape':   # Esc critic input order: obs_1_own, act_1_own, obs_2, act_2
        return dense(q['val_out'], _sh(p, jnp.tanh(dense(q['inp1_val'], jnp.concatenate([c1, c2], -1)))))[..., 0]
    h = jnp.tanh(dense(q['c2'], jnp.tanh(dense(q['c1'], jnp.concatenate([c1, c2], -1)))))
    return dense(q['val_out'], h)[..., 0]


def low_train_window(family, p, w, mb, seq):
    """Training forward over one window (W rows) -> logits [W,L], V [W], pad outputs or None.
    seq=True: RLlib temporal attention over the chunk incl. future steps + padding emulation."""
    q, d = p[f'ac{w}'], LOW_DIMS[w]
    obs = mb['own']
    if not (family == 'fight_sa' and seq):
        logits = low_actor_infer(family, p, w, obs)
        V = low_critic_infer(family, p, w, obs, mb['own_a'], mb['oth'], mb['oth_a'])
        return logits, V, None
    W = obs.shape[0]
    allowed = mb['ep'][:, None] == mb['ep'][None, :]
    Lseg = jnp.sum(allowed, 1).astype(jnp.float32)
    n_pad = W - Lseg
    D, ss = d['Df'], d['ss']
    a, b = jnp.tanh(dense(q['inp1'], obs[..., :ss])), jnp.tanh(dense(q['inp2'], obs[..., ss:D]))
    f = jnp.tanh(dense(q['inp3'], obs[..., :D]))
    fp = jnp.tanh(q['inp3']['b'])
    att, att_p = mha_window(q['att'], f, allowed, n_pad, fp)
    logits = dense(q['act_out'], _sh(p, jnp.concatenate([a, b, l2n(f + att)], -1)))
    ap, bp = jnp.tanh(q['inp1']['b']), jnp.tanh(q['inp2']['b'])
    logits_p = dense(q['act_out'], _sh(p, jnp.concatenate([jnp.broadcast_to(ap, (W, 200)),
                                                          jnp.broadcast_to(bp, (W, 200)), l2n(fp + att_p)], -1)))
    c1, c2 = _critic_parts(family, w, obs, mb['own_a'], mb['oth'], mb['oth_a'])
    y = jnp.tanh(dense(q['v3'], jnp.concatenate([c1, c2], -1)))
    yp = jnp.tanh(q['v3']['b'])
    vatt, vatt_p = mha_window(q['att_val'], y, allowed, n_pad, yp)
    V = dense(q['val_out'], _sh(p, jnp.concatenate([jnp.tanh(dense(q['v1'], c1)), jnp.tanh(dense(q['v2'], c2)),
                                                   l2n(y + vatt)], -1)))[..., 0]
    v1p, v2p = jnp.tanh(q['v1']['b']), jnp.tanh(q['v2']['b'])
    Vp = dense(q['val_out'], _sh(p, jnp.concatenate([jnp.broadcast_to(v1p, (W, 175)), jnp.broadcast_to(v2p, (W, 175)),
                                                    l2n(yp + vatt_p)], -1)))[..., 0]
    return logits, V, dict(logits=logits_p, V=Vp, w=n_pad / Lseg)


def frozen_policy_actions(kind, p, obs, ac1):
    """Deterministic argmax actions of a frozen low-level policy for rows with mixed AC types."""
    fam = 'fight_sa' if kind == 'fight' else 'escape'
    a1 = md_argmax(low_actor_infer(fam, p, 1, obs), SPLITS1)
    a2 = md_argmax(low_actor_infer(fam, p, 2, obs), SPLITS2)
    return jnp.where(ac1[:, None], a1, a2).astype(jnp.int32)


def frozen_ll_actions(p_fight, p_esc, obs_f, obs_e, ac1, use_esc):
    af = frozen_policy_actions('fight', p_fight, obs_f, ac1)
    ae = frozen_policy_actions('escape', p_esc, obs_e, ac1)
    return jnp.where(use_esc[:, None], ae, af)


# ------------------------------------------------------------------ commander networks
H_CMD = 200


def cmd_init(family, key):
    k = jax.random.split(key, 14)
    p = {'shared': dense_init(k[0], 500, 500) if family != 'cmd_fc' else {}}
    if family == 'cmd_fc':
        p['pi'] = dict(a1=dense_init(k[1], 34, 500), a2=dense_init(k[2], 500, 500), act_out=dense_init(k[3], 500, 3),
                       c1=dense_init(k[4], 105, 500), c2=dense_init(k[5], 500, 500), val_out=dense_init(k[6], 500, 1))
        return p
    mem = (lambda kk: gru_init(kk, 200, 200)) if family == 'cmd_gru' else (lambda kk: mha_init(kk, 200))
    p['pi'] = dict(inp1=dense_init(k[1], 4, 50), inp2=dense_init(k[2], 20, 200), inp3=dense_init(k[3], 10, 50),
                   inp4=dense_init(k[4], 34, 200), mem_act=mem(k[5]), act_out=dense_init(k[6], 500, 3),
                   v1=dense_init(k[7], 35, 100), v2=dense_init(k[8], 35, 100), v3=dense_init(k[9], 35, 100),
                   v4=dense_init(k[10], 105, 200), mem_val=mem(k[11]), val_out=dense_init(k[12], 500, 1))
    return p


def _cmd_actor_parts(q, o):
    x = jnp.concatenate([jnp.tanh(dense(q['inp1'], o[..., :4])), jnp.tanh(dense(q['inp2'], o[..., 4:24])),
                         jnp.tanh(dense(q['inp3'], o[..., 24:34]))], -1)
    return x, jnp.tanh(dense(q['inp4'], o))


def _cmd_critic_parts(q, cin):
    z = jnp.concatenate([jnp.tanh(dense(q['v1'], cin[..., 0:35])), jnp.tanh(dense(q['v2'], cin[..., 35:70])),
                         jnp.tanh(dense(q['v3'], cin[..., 70:105]))], -1)
    return z, jnp.tanh(dense(q['v4'], cin))


def cmd_step(family, p, o, cin, h):
    """One rollout step. o [...,34], cin [...,105] = [o1,a1,o2,a2,o3,a3], h [...,2,200] -> logits, V, h'."""
    q = p['pi']
    if family == 'cmd_fc':
        lg = dense(q['act_out'], jnp.tanh(dense(q['a2'], jnp.tanh(dense(q['a1'], o)))))
        V = dense(q['val_out'], jnp.tanh(dense(q['c2'], jnp.tanh(dense(q['c1'], cin)))))[..., 0]
        return lg, V, h
    x, f = _cmd_actor_parts(q, o)
    z, g = _cmd_critic_parts(q, cin)
    if family == 'cmd_gru':
        ha, hv = gru_cell(q['mem_act'], f, h[..., 0, :]), gru_cell(q['mem_val'], g, h[..., 1, :])
        fa, gv = ha, hv
        h = jnp.stack([ha, hv], -2)
    else:
        fa, gv = mha_single(q['mem_act'], f), mha_single(q['mem_val'], g)
    lg = dense(q['act_out'], _sh(p, jnp.concatenate([x, l2n(f + fa)], -1)))
    V = dense(q['val_out'], _sh(p, jnp.concatenate([z, l2n(g + gv)], -1)))[..., 0]
    return lg, V, h


def cmd_train_window(family, p, mb, seq):
    """Training forward over one window of W commander steps of one agent."""
    q = p['pi']
    o, cin = mb['obs'], mb['cin']
    W = o.shape[0]
    if family == 'cmd_fc' or (family == 'cmd_sa' and not seq):
        lg, V, _ = cmd_step(family, p, o, cin, jnp.zeros((W, 2, H_CMD)))
        return lg, V, None
    x, f = _cmd_actor_parts(q, o)
    z, g = _cmd_critic_parts(q, cin)
    if family == 'cmd_gru':
        newep = jnp.concatenate([jnp.zeros((1,), bool), mb['ep'][1:] != mb['ep'][:-1]])

        def body(h, inp):
            fi, gi, reset = inp
            h = jnp.where(reset, 0.0, h)
            ha, hv = gru_cell(q['mem_act'], fi, h[0]), gru_cell(q['mem_val'], gi, h[1])
            return jnp.stack([ha, hv]), (ha, hv)
        _, (fa, gv) = jax.lax.scan(body, mb['h0'], (f, g, newep))
        lg = dense(q['act_out'], _sh(p, jnp.concatenate([x, l2n(f + fa)], -1)))
        V = dense(q['val_out'], _sh(p, jnp.concatenate([z, l2n(g + gv)], -1)))[..., 0]
        return lg, V, None
    # cmd_sa with RLlib chunk attention + padding emulation
    allowed = mb['ep'][:, None] == mb['ep'][None, :]
    Lseg = jnp.sum(allowed, 1).astype(jnp.float32)
    n_pad = W - Lseg
    fp, gp = jnp.tanh(q['inp4']['b']), jnp.tanh(q['v4']['b'])
    fa, fa_p = mha_window(q['mem_act'], f, allowed, n_pad, fp)
    gv, gv_p = mha_window(q['mem_val'], g, allowed, n_pad, gp)
    lg = dense(q['act_out'], _sh(p, jnp.concatenate([x, l2n(f + fa)], -1)))
    V = dense(q['val_out'], _sh(p, jnp.concatenate([z, l2n(g + gv)], -1)))[..., 0]
    xp = jnp.concatenate([jnp.tanh(q['inp1']['b']), jnp.tanh(q['inp2']['b']), jnp.tanh(q['inp3']['b'])])
    zp = jnp.concatenate([jnp.tanh(q['v1']['b']), jnp.tanh(q['v2']['b']), jnp.tanh(q['v3']['b'])])
    lgp = dense(q['act_out'], _sh(p, jnp.concatenate([jnp.broadcast_to(xp, (W, 300)), l2n(fp + fa_p)], -1)))
    Vp = dense(q['val_out'], _sh(p, jnp.concatenate([jnp.broadcast_to(zp, (W, 300)), l2n(gp + gv_p)], -1)))[..., 0]
    return lg, V, dict(logits=lgp, V=Vp, w=n_pad / Lseg)


# ------------------------------------------------------------------ PPO pieces
def gae(rew, val, done, last_val, gamma, lam):
    """rew/val: [T, ...], done: [T, B] (broadcast), last_val: [...]."""
    d = done.reshape(done.shape + (1,) * (rew.ndim - done.ndim)).astype(jnp.float32)

    def f(carry, x):
        adv_n, v_n = carry
        r, v, dn = x
        nt = 1.0 - dn
        delta = r + gamma * v_n * nt - v
        adv = delta + gamma * lam * nt * adv_n
        return (adv, v), adv
    _, adv = jax.lax.scan(f, (jnp.zeros_like(last_val), last_val), (rew, val, d), reverse=True)
    return adv, adv + val


def ppo_loss_terms(logits, V, pad, mb, splits, hp, klc, masked_mean=False):
    lp = md_logp(logits, mb['act'], splits)
    ratio = jnp.exp(lp - mb['logp'])
    adv = mb['adv']
    surr = jnp.minimum(adv * ratio, adv * jnp.clip(ratio, 1 - hp['clip'], 1 + hp['clip']))
    vf = jnp.clip((V - mb['vt']) ** 2, 0.0, hp['vf_clip'])
    kl = md_kl(mb['logits'], logits, splits)
    ent = md_entropy(logits, splits)
    if pad is None:
        mean = lambda x: jnp.mean(x)
        pi_l, vf_l, kl_m, ent_m = -mean(surr), mean(vf), mean(kl), mean(ent)
    else:  # unmasked mean over real + RLlib padded rows
        pw = pad['w']
        den = jnp.sum(1.0 + pw)
        klp = md_kl(jnp.zeros_like(pad['logits']), pad['logits'], splits)
        vfp = jnp.clip(pad['V'] ** 2, 0.0, hp['vf_clip'])
        entp = md_entropy(pad['logits'], splits)
        pi_l = -jnp.sum(surr) / den
        vf_l = (jnp.sum(vf) + jnp.sum(pw * vfp)) / den
        kl_m = (jnp.sum(kl) + jnp.sum(pw * klp)) / den
        ent_m = (jnp.sum(ent) + jnp.sum(pw * entp)) / den
    total = pi_l + hp['vf_coeff'] * vf_l - hp['ent_coeff'] * ent_m + klc * kl_m
    return total, dict(pi_loss=pi_l, vf_loss=vf_l, kl=kl_m, entropy=ent_m)


def to_windows(x, W):
    """[T, B, ...] -> [(T/W)*B, W, ...]"""
    T, B = x.shape[:2]
    x = x.reshape((T // W, W, B) + x.shape[2:])
    x = jnp.swapaxes(x, 1, 2)
    return x.reshape(((T // W) * B, W) + x.shape[3:])


def kl_adapt(klc, kl, target):
    return jnp.where(kl > 2.0 * target, klc * 1.5, jnp.where(kl < 0.5 * target, klc * 0.5, klc))


def sgd_epochs(loss_fn, opt, params, sub_keys, opt_state, data, key, n_epochs, n_mb, mbw):
    """RLlib do_minibatch_sgd for one policy: epochs x shuffled minibatches of windows, one Adam step each.
    Only the parameter subtrees `sub_keys` are trained (e.g. ('shared', 'ac1'))."""
    nwin = jax.tree_util.tree_leaves(data)[0].shape[0]

    def mb_step(carry, idx):
        params, os = carry
        mb = jax.tree_util.tree_map(lambda x: x[idx], data)
        sub = {k: params[k] for k in sub_keys}
        (_, aux), g = jax.value_and_grad(lambda sp: loss_fn({**params, **sp}, mb), has_aux=True)(sub)
        upd, os = opt.update(g, os, sub)
        params = {**params, **optax.apply_updates(sub, upd)}
        return (params, os), aux

    def epoch(carry, k):
        perm = jax.random.permutation(k, nwin)[:n_mb * mbw].reshape(n_mb, mbw)
        return jax.lax.scan(mb_step, carry, perm)
    (params, opt_state), aux = jax.lax.scan(epoch, (params, opt_state), jax.random.split(key, n_epochs))
    last = jax.tree_util.tree_map(lambda x: jnp.mean(x[-1]), aux)       # stats of the last epoch
    return params, opt_state, last


def standardize(x):
    return (x - jnp.mean(x)) / jnp.maximum(jnp.std(x), 1e-4)

### Trainers
`LowTrainer` trains the two low-level policies (`ac1_policy` drives agent 1 (AC1) and `ac2_policy` drives agent 2 (AC2)) in
the 2-vs-2 env. Each policy has a centralized critic over [own obs, own act, partner obs, partner act].
`CmdTrainer` trains the shared commander policy for 3 agents (3 vs 3) with critic input [own, other1, other2] (obs + act).
One call of `train_iter` = one PPO iteration (rollout of `n_envs × rollout` env steps, GAE, all SGD epochs).

In [ ]:
class LowTrainer:
    def __init__(self, family, level, mode, hp, n_envs, rollout, minibatch, epochs, faithful=RLLIB_FAITHFUL,
                 opp_pool=None):
        self.family, self.hp, self.faithful = family, hp, faithful
        self.env = Env('low', level=level, mode=mode, opp_pool=opp_pool)
        self.reset_b, self.step_b = make_batched(self.env)
        self.B, self.Tr = n_envs, rollout
        self.seq = faithful and family == 'fight_sa'
        self.W = hp['max_seq_len'] if self.seq else 1
        assert rollout % self.W == 0, 'rollout must be a multiple of max_seq_len (20)'
        nwin = (rollout // self.W) * n_envs
        self.mbw = max(1, min(nwin, minibatch // self.W))
        self.n_mb = max(1, nwin // self.mbw)
        self.epochs = epochs
        self.opt = optax.adam(hp['lr'], eps=1e-8)
        self.train_iter = jax.jit(self._train_iter)

    def init(self, key, params=None):
        k1, k2 = jax.random.split(key)
        params = params if params is not None else low_init(self.family, k1)
        opt_state = {w: self.opt.init({'shared': params['shared'], w: params[w]}) for w in ('ac1', 'ac2')}
        klc = {w: jnp.float32(self.hp['kl_coeff']) for w in ('ac1', 'ac2')}
        return dict(params=params, opt=opt_state, klc=klc, runner=self.init_runner(k2))

    def init_runner(self, key):
        k1, k2 = jax.random.split(key)
        s, obs = self.reset_b(jax.random.split(k1, self.B))
        return dict(s=s, obs=obs, key=k2, ep=jnp.arange(self.B, dtype=jnp.int32) * 0,
                    ret=jnp.zeros(self.B), n_ep=jnp.zeros(self.B, jnp.int32))

    def _values(self, params, obs, act_n):
        f = self.family
        return jnp.stack([low_critic_infer(f, params, 1, obs[:, 0], act_n[:, 0], obs[:, 1], act_n[:, 1]),
                          low_critic_infer(f, params, 2, obs[:, 1], act_n[:, 1], obs[:, 0], act_n[:, 0])], 1)

    def _rollout(self, params, runner, opp_params):
        f, B = self.family, self.B

        def step(c, _):
            s, obs, key, ep, ret = c
            key, k1, k2, k3, k4 = jax.random.split(key, 5)
            l1 = low_actor_infer(f, params, 1, obs[:, 0])
            l2 = low_actor_infer(f, params, 2, obs[:, 1])
            a1, a2 = md_sample(k1, l1, SPLITS1), md_sample(k2, l2, SPLITS2)
            act = jnp.stack([a1, a2], 1)
            logp = jnp.stack([md_logp(l1, a1, SPLITS1), md_logp(l2, a2, SPLITS2)], 1)
            v = self._values(params, obs, jnp.zeros((B, 2, 4)))       # rollout critic: zero action inputs
            s2, obs2, r, d, info, _ = self.step_b(s, act, jax.random.split(k3, B), opp_params)
            ret2 = ret + r.sum(1)
            sr, obr = self.reset_b(jax.random.split(k4, B))
            tr = dict(obs=obs, act=act, logp=logp, v=v, l1=l1, l2=l2, rew=r, done=d, ep=ep)
            m = dict(ret=jnp.where(d, ret2, 0.0), n=d.astype(jnp.float32), win=(info['win'] & d).astype(jnp.float32),
                     loss=(info['loss'] & d).astype(jnp.float32), draw=(info['draw'] & d).astype(jnp.float32),
                     len=jnp.where(d, s2['steps'], 0).astype(jnp.float32))
            return (tree_where(d, sr, s2), jnp.where(d[:, None, None], obr, obs2), key, ep + d,
                    jnp.where(d, 0.0, ret2)), (tr, m)
        c0 = (runner['s'], runner['obs'], runner['key'], runner['ep'], runner['ret'])
        (s, obs, key, ep, ret), (tr, m) = jax.lax.scan(step, c0, None, length=self.Tr)
        last_v = self._values(params, obs, jnp.zeros((B, 2, 4)))
        runner = dict(runner, s=s, obs=obs, key=key, ep=ep, ret=ret)
        return runner, tr, last_v, jax.tree_util.tree_map(jnp.sum, m)

    def _train_iter(self, st, opp_params, key):
        hp = self.hp
        params = st['params']
        runner, tr, last_v, m = self._rollout(params, st['runner'], opp_params)
        adv, vt = gae(tr['rew'], tr['v'], tr['done'], last_v, hp['gamma'], hp['lam'])
        actn = tr['act'] / ACT_NORM                                     # normalized actions for the critic
        if not self.faithful:
            actn = jnp.zeros_like(actn)
        opt_state, klc, stats = dict(st['opt']), dict(st['klc']), {}
        ks = jax.random.split(key, 2)
        for wi, w in enumerate(('ac1', 'ac2')):
            o = 1 - wi
            data = dict(own=tr['obs'][:, :, wi], own_a=actn[:, :, wi], oth=tr['obs'][:, :, o], oth_a=actn[:, :, o],
                        act=tr['act'][:, :, wi], logp=tr['logp'][:, :, wi], vt=vt[:, :, wi],
                        adv=standardize(adv[:, :, wi]), logits=tr['l1' if wi == 0 else 'l2'], ep=tr['ep'])
            data = jax.tree_util.tree_map(lambda x: to_windows(x, self.W), data)
            splits = SPLITS1 if wi == 0 else SPLITS2

            def loss_fn(p, mb, wi=wi, splits=splits, w=w):
                out = jax.vmap(lambda m_: low_train_window(self.family, p, wi + 1, m_, self.seq))(mb)
                return ppo_loss_terms(out[0], out[1], out[2], mb, splits, hp, klc[w])
            params, opt_state[w], aux = sgd_epochs(loss_fn, self.opt, params, ('shared', w), opt_state[w], data,
                                                   ks[wi], self.epochs, self.n_mb, self.mbw)
            klc[w] = kl_adapt(klc[w], aux['kl'], hp['kl_target'])
            stats.update({f'{k}_{w}': v for k, v in aux.items()})
            stats[f'klc_{w}'] = klc[w]
        stats.update(m)
        return dict(params=params, opt=opt_state, klc=klc, runner=runner), stats


class CmdTrainer:
    """Commander (3 vs 3) PPO with one policy shared by the 3 agents."""

    def __init__(self, family, hp, n_envs, rollout, minibatch, epochs, faithful=RLLIB_FAITHFUL):
        self.family, self.hp, self.faithful = family, hp, faithful
        self.env = Env('high', n=3, m=3)
        self.reset_b, self.step_b = make_batched(self.env)
        self.B, self.Tr = n_envs, rollout
        self.seq = faithful and family == 'cmd_sa'
        self.W = hp['max_seq_len'] if family in ('cmd_gru', 'cmd_sa') else 1
        assert rollout % self.W == 0
        nwin = (rollout // self.W) * n_envs * 3
        self.mbw = max(1, min(nwin, minibatch // self.W))
        self.n_mb = max(1, nwin // self.mbw)
        self.epochs = epochs
        self.opt = optax.adam(hp['lr'], eps=1e-8)
        self.train_iter = jax.jit(self._train_iter)
        self.others = np.array([[1, 2], [0, 2], [0, 1]])

    def init(self, key, params=None):
        k1, k2 = jax.random.split(key)
        params = params if params is not None else cmd_init(self.family, k1)
        return dict(params=params, opt=self.opt.init(params), klc=jnp.float32(self.hp['kl_coeff']),
                    runner=self.init_runner(k2))

    def init_runner(self, key):
        k1, k2 = jax.random.split(key)
        s, obs = self.reset_b(jax.random.split(k1, self.B))
        return dict(s=s, obs=obs, key=k2, ep=jnp.zeros(self.B, jnp.int32), ret=jnp.zeros(self.B),
                    h=jnp.zeros((self.B, 3, 2, H_CMD)))

    def cin(self, obs, act_n):
        """critic input per agent: [o_own, a_own, o_other1, a_other1, o_other2, a_other2] -> [B,3,105]"""
        rows = []
        for i in range(3):
            parts = [obs[:, i], act_n[:, i, None]]
            for j in self.others[i]:
                parts += [obs[:, j], act_n[:, j, None]]
            rows.append(jnp.concatenate(parts, -1))
        return jnp.stack(rows, 1)

    def _rollout(self, params, runner, llp):
        f, B = self.family, self.B

        def step(c, _):
            s, obs, key, ep, ret, h = c
            key, k1, k3, k4 = jax.random.split(key, 4)
            lg, v, h2 = cmd_step(f, params, obs, self.cin(obs, jnp.zeros((B, 3))), h)
            a = jax.random.categorical(k1, lg)
            logp = jnp.take_along_axis(jax.nn.log_softmax(lg), a[..., None], -1)[..., 0]
            s2, obs2, r, d, info, _ = self.step_b(s, a, jax.random.split(k3, B), llp)
            ret2 = ret + r.sum(1)
            sr, obr = self.reset_b(jax.random.split(k4, B))
            tr = dict(obs=obs, act=a, logp=logp, v=v, logits=lg, rew=r, done=d, ep=ep, h=h)
            m = dict(ret=jnp.where(d, ret2, 0.0), n=d.astype(jnp.float32), win=(info['win'] & d).astype(jnp.float32),
                     loss=(info['loss'] & d).astype(jnp.float32), draw=(info['draw'] & d).astype(jnp.float32),
                     len=jnp.where(d, s2['steps'], 0).astype(jnp.float32))
            h2 = jnp.where(d[:, None, None, None], 0.0, h2)
            return (tree_where(d, sr, s2), jnp.where(d[:, None, None], obr, obs2), key, ep + d,
                    jnp.where(d, 0.0, ret2), h2), (tr, m)
        c0 = (runner['s'], runner['obs'], runner['key'], runner['ep'], runner['ret'], runner['h'])
        (s, obs, key, ep, ret, h), (tr, m) = jax.lax.scan(step, c0, None, length=self.Tr)
        _, last_v, _ = cmd_step(f, params, obs, self.cin(obs, jnp.zeros((B, 3))), h)
        runner = dict(runner, s=s, obs=obs, key=key, ep=ep, ret=ret, h=h)
        return runner, tr, last_v, jax.tree_util.tree_map(jnp.sum, m)

    def _train_iter(self, st, llp, key):
        hp, W = self.hp, self.W
        params = st['params']
        runner, tr, last_v, m = self._rollout(params, st['runner'], llp)
        adv, vt = gae(tr['rew'], tr['v'], tr['done'], last_v, hp['gamma'], hp['lam'])
        Tn, B = tr['act'].shape[:2]
        actn = tr['act'].astype(jnp.float32) / 2.0
        if not self.faithful:
            actn = jnp.zeros_like(actn)
        cin = jax.vmap(self.cin)(tr['obs'], actn)                         # [T,B,3,105]
        flat = lambda x: x.reshape((Tn, B * 3) + x.shape[3:])
        data = dict(obs=flat(tr['obs']), cin=flat(cin), act=flat(tr['act']), logp=flat(tr['logp']),
                    vt=flat(vt), adv=standardize(flat(adv)), logits=flat(tr['logits']),
                    ep=flat(jnp.repeat(tr['ep'][:, :, None], 3, 2)), h0=flat(tr['h']))
        data = jax.tree_util.tree_map(lambda x: to_windows(x, W), data)
        data['h0'] = data['h0'][:, 0]                                    # hidden state at the window start
        klc = st['klc']

        def loss_fn(p, mb):
            out = jax.vmap(lambda m_: cmd_train_window(self.family, p, m_, self.seq))(mb)
            mb2 = dict(mb, act=mb['act'][..., None])
            return ppo_loss_terms(out[0], out[1], out[2], mb2, (3,), hp, klc)
        keys = tuple(params.keys())
        params, opt_state, aux = sgd_epochs(loss_fn, self.opt, params, keys, st['opt'], data, key,
                                            self.epochs, self.n_mb, self.mbw)
        klc = kl_adapt(klc, aux['kl'], hp['kl_target'])
        stats = dict(aux, klc=klc, **m)
        return dict(params=params, opt=opt_state, klc=klc, runner=runner), stats

## 5. Training runs (paper experiments)

**Stages.** Each stage is one original training run:

| stage | original command | paper | init from | opponents |
|---|---|---|---|---|
| `fight_L1`, `fight_L2`, `fight_L3` | `train_hetero.py --level 1..3` | Fig. 6 "SA-Net" (curriculum) | previous level (full algo state) | static / random / scripted |
| `fight_L3_nocurr` | level 3 from scratch | Fig. 6 "SA-Net (no Curr)" | – | scripted |
| `fight_L3_fc` | FC baseline | Fig. 6 "FC" | – (or FC L1→L2 if `FIG6_FC_CURRICULUM`) | scripted |
| `escape_L3` | `--agent_mode escape` | Fig. 8a | – | scripted L3 |
| `fight_L4` | `--level 4` | – | `fight_L3` | frozen `fight_L3` |
| `fight_L5` | `--level 5` | – | `fight_L4` | per episode one of {L3 fight, L4 fight, L3 escape} (paper: L1..L4) |
| `escape_L5` (optional) | escape at L5 | – | `escape_L3` | frozen `fight_L5` |
| `commander_gru` / `_sa` / `_fc` | `train_hier.py` | Fig. 10a | – | low level: L5 fight + escape |

Dependencies are trained automatically. After every stage, the following are written to `OUT_DIR`:
* `checkpoints/<stage>.pkl`: params, both Adam states, KL coefficients and counters. It is also written periodically
  and when the session time budget runs out.
* `policies/<stage>.pkl`: the frozen parameters used by later stages, like the original `policies/L*_AC*_*.pt`.
* `logs/<stage>.csv`: one row per PPO iteration.
* `status.json`.

A stage that is already finished is skipped; an unfinished one resumes from its checkpoint (a fresh env batch is
drawn on resume). To continue on a new Kaggle session, add the previous version's output as an input dataset: every
directory in `RESUME_DIRS` that contains a `status.json` is merged into `OUT_DIR` first.

In [ ]:
def save_pickle(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(jax.device_get(obj), f)
    os.replace(tmp, path)


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


STATUS_PATH = os.path.join(OUT_DIR, 'status.json')


def load_status():
    try:
        with open(STATUS_PATH) as f:
            return json.load(f)
    except Exception:
        return {}


def save_status(st):
    with open(STATUS_PATH + '.tmp', 'w') as f:
        json.dump(st, f, indent=1)
    os.replace(STATUS_PATH + '.tmp', STATUS_PATH)


def merge_resume_dirs():
    """Copy checkpoints/policies/logs/eval of previous runs (Kaggle input datasets) into OUT_DIR."""
    status = load_status()
    for d in RESUME_DIRS:
        sp = os.path.join(d, 'status.json')
        if not os.path.isfile(sp) or os.path.abspath(d) == os.path.abspath(OUT_DIR):
            continue
        with open(sp) as f:
            other = json.load(f)
        print('resuming from', d, '->', {k: v.get('env_steps') for k, v in other.items()})
        for sub in ('checkpoints', 'policies', 'logs', 'eval'):
            for src in glob.glob(os.path.join(d, sub, '*')):
                dst = os.path.join(OUT_DIR, sub, os.path.basename(src))
                name = os.path.splitext(os.path.basename(src))[0]
                newer = other.get(name, {}).get('env_steps', 0) > status.get(name, {}).get('env_steps', -1)
                if not os.path.exists(dst) or (sub != 'eval' and newer):
                    shutil.copy2(src, dst)
        for k, v in other.items():
            if v.get('env_steps', 0) > status.get(k, {}).get('env_steps', -1) or (v.get('done') and not status.get(k, {}).get('done')):
                status[k] = v
    save_status(status)


merge_resume_dirs()


def time_left_h():
    return MAX_SESSION_HOURS - (time.time() - SESSION_START) / 3600.0


# ------------------------------------------------------------------ stage graph
def _fight(level, init, opps=(), family='fight_sa'):
    return dict(kind='low', family=family, level=level, mode='fight', init=init, opps=tuple(opps))


L5_POOL = ('fight_L1', 'fight_L2', 'fight_L3', 'fight_L4') if PAPER_L5_POOL else ('fight_L3', 'fight_L4', 'escape_L3')
STAGES = {
    'fight_L1': _fight(1, None), 'fight_L2': _fight(2, 'fight_L1'), 'fight_L3': _fight(3, 'fight_L2'),
    'fight_L3_nocurr': _fight(3, None),
    'fc_L1': _fight(1, None, family='fight_fc'), 'fc_L2': _fight(2, 'fc_L1', family='fight_fc'),
    'fight_L3_fc': _fight(3, 'fc_L2' if FIG6_FC_CURRICULUM else None, family='fight_fc'),
    'escape_L3': dict(kind='low', family='escape', level=3, mode='escape', init=None, opps=()),
    'fight_L4': _fight(4, 'fight_L3', ('fight_L3',)),
    'fight_L5': _fight(5, 'fight_L4', L5_POOL),
    'escape_L5': dict(kind='low', family='escape', level=5, mode='escape', init='escape_L3', opps=('fight_L5',)),
}
ESC_FOR_HL = 'escape_L5' if ESCAPE_L5_STAGE else 'escape_L3'
for _fam in ('gru', 'sa', 'fc'):
    STAGES[f'commander_{_fam}'] = dict(kind='cmd', family=f'cmd_{_fam}', init=None, opps=(),
                                       ll=dict(fight='fight_L5', escape=ESC_FOR_HL))
STAGE_BUDGET.setdefault('fc_L1', STAGE_BUDGET['fight_L1'])
STAGE_BUDGET.setdefault('fc_L2', STAGE_BUDGET['fight_L2'])


def stage_deps(name):
    sp = STAGES[name]
    deps = ([sp['init']] if sp.get('init') else []) + list(sp.get('opps', ()))
    if sp['kind'] == 'cmd':
        deps += list(sp['ll'].values())
    return deps


def load_policy(name):
    return jax.tree_util.tree_map(jnp.asarray, load_pickle(os.path.join(DIRS['policies'], name + '.pkl')))


def ll_policies(fight_ag, fight_opp=None, escape=ESC_FOR_HL):
    return dict(fight_ag=load_policy(fight_ag), fight_opp=load_policy(fight_opp or fight_ag), escape=load_policy(escape))


def make_trainer(name):
    sp = STAGES[name]
    if sp['kind'] == 'low':
        env_pool = None
        if sp['level'] == 5 and sp['mode'] == 'escape':
            env_pool = ('fight',)
        elif sp['level'] == 5:
            env_pool = tuple('escape' if o.startswith('escape') else 'fight' for o in sp['opps'])
        tr = LowTrainer(sp['family'], sp['level'], sp['mode'], PPO_LOW, HP['low_envs'], HP['low_rollout'],
                        HP['low_minibatch'], HP['low_epochs'], opp_pool=env_pool)
        opp = tuple(load_policy(o) for o in sp['opps'])
        return tr, opp
    tr = CmdTrainer(sp['family'], PPO_CMD, HP['cmd_envs'], HP['cmd_rollout'], HP['cmd_minibatch'], HP['cmd_epochs'])
    return tr, ll_policies(sp['ll']['fight'], escape=sp['ll']['escape'])


class SessionTimeout(Exception):
    pass


def run_stage(name, quiet=False):
    status = load_status()
    if status.get(name, {}).get('done'):
        if not quiet:
            print(f'[{name}] already done ({status[name]["env_steps"]:,} env steps), skipped')
        return
    for d in stage_deps(name):
        run_stage(d, quiet=True)
    budget = STAGE_BUDGET[name]
    tr, opp = make_trainer(name)
    per_iter = tr.B * tr.Tr
    ck_path = os.path.join(DIRS['checkpoints'], name + '.pkl')
    log_path = os.path.join(DIRS['logs'], name + '.csv')
    key = jax.random.PRNGKey(SEED + sum(map(ord, name)))
    if os.path.exists(ck_path):
        ck = load_pickle(ck_path)
        st = tr.init(jax.random.fold_in(key, ck['iter']), params=jax.tree_util.tree_map(jnp.asarray, ck['params']))
        st['opt'], st['klc'] = jax.tree_util.tree_map(jnp.asarray, (ck['opt'], ck['klc']))
        it, env_steps, wall = ck['iter'], ck['env_steps'], ck['wall']
        print(f'[{name}] resuming at iter {it}, {env_steps:,} env steps')
    else:
        st = tr.init(key)
        it, env_steps, wall = 0, 0, 0.0
        parent = STAGES[name].get('init')
        if parent:   # curriculum transfer: restore the full algorithm state (params, Adam, KL coeff) like algo.restore
            pk = load_pickle(os.path.join(DIRS['checkpoints'], parent + '.pkl'))
            st['params'], st['opt'], st['klc'] = jax.tree_util.tree_map(jnp.asarray, (pk['params'], pk['opt'], pk['klc']))
            print(f'[{name}] initialized from {parent}')
        if os.path.exists(log_path):
            os.remove(log_path)

    def checkpoint(done=False):
        save_pickle(ck_path, dict(params=st['params'], opt=st['opt'], klc=st['klc'], iter=it, env_steps=env_steps,
                                  wall=wall))
        s_ = load_status()
        s_[name] = dict(done=done, env_steps=int(env_steps), iters=int(it), wall_s=round(wall, 1),
                        preset=PRESET, budget=int(budget))
        save_status(s_)

    n_iters = max(1, -(-(budget - env_steps) // per_iter))
    print(f'[{name}] {tr.B} envs x {tr.Tr} steps = {per_iter} env steps/iter, {n_iters} iters to go '
          f'(budget {budget:,}), {tr.epochs} epochs x {tr.n_mb} minibatches of {tr.mbw}x{tr.W} rows')
    last_ck, t_print, header = time.time(), 0.0, not os.path.exists(log_path)
    with open(log_path, 'a', newline='') as fcsv:
        writer = None
        while env_steps < budget:
            if time_left_h() <= 0:
                checkpoint(False)
                print(f'[{name}] session time budget exhausted: checkpoint saved at {env_steps:,} env steps')
                raise SessionTimeout(name)
            t0 = time.time()
            st, stats = tr.train_iter(st, opp, jax.random.fold_in(key, it + 1))
            stats = {k: float(v) for k, v in jax.device_get(stats).items()}
            dt = time.time() - t0
            it += 1; env_steps += per_iter; wall += dt
            n = max(stats.pop('n'), 0.0)
            row = dict(iter=it, env_steps=env_steps, wall_s=round(wall, 2), sps=round(per_iter / dt, 1),
                       episodes=n, ep_return=stats.pop('ret') / n if n else float('nan'),
                       ep_len=stats.pop('len') / n if n else float('nan'),
                       win=stats.pop('win') / n if n else float('nan'), loss=stats.pop('loss') / n if n else float('nan'),
                       draw=stats.pop('draw') / n if n else float('nan'))
            row.update({k: round(v, 6) for k, v in stats.items()})
            if writer is None:
                writer = csv.DictWriter(fcsv, fieldnames=list(row.keys()))
                if header:
                    writer.writeheader()
            writer.writerow(row); fcsv.flush()
            if time.time() - t_print > 60 or env_steps >= budget or it == 1:
                t_print = time.time()
                print(f'[{name}] it {it:6d} | {env_steps:>11,} steps | {row["sps"]:>9,.0f} sps | return '
                      f'{row["ep_return"]:7.3f} | win {row["win"]:.2f} loss {row["loss"]:.2f} | {time_left_h():.2f} h left')
            if time.time() - last_ck > CKPT_EVERY_MIN * 60:
                checkpoint(False); last_ck = time.time()
    checkpoint(True)
    save_pickle(os.path.join(DIRS['policies'], name + '.pkl'), st['params'])
    print(f'[{name}] done: {env_steps:,} env steps in {wall / 60:.1f} min')


EXPERIMENT_STAGES = {
    'fig6': ['fight_L1', 'fight_L2', 'fight_L3', 'fight_L3_nocurr', 'fight_L3_fc'],
    'fig8': ['escape_L3'],
    'fsp': ['fight_L4', 'fight_L5'] + (['escape_L5'] if ESCAPE_L5_STAGE else []),
    'fig10': ['commander_gru', 'commander_sa', 'commander_fc'],
}
STOPPED = False


def run_experiments():
    global STOPPED
    for ex in EXPERIMENTS:
        for name in EXPERIMENT_STAGES.get(ex, []):
            try:
                run_stage(name)
            except SessionTimeout:
                STOPPED = True
                print('Stopping: re-run the notebook in a new session (with this output attached) to continue.')
                return

### Run the training stages
This cell trains everything selected in `EXPERIMENTS` (it is the long-running cell). Progress is printed about
once per minute; the full curves are in `OUT_DIR/logs/*.csv`.

In [ ]:
run_experiments()
print(json.dumps(load_status(), indent=1))

## 6. Evaluation

Like `evaluation.py`, every evaluation runs `EVAL_EPISODES` episodes of the **commander env** (50 km map,
horizon 500) in n-vs-n, with all episodes running in parallel.

* **Fig. 7b / 7a, low-level fight** (`eval_hl=False`):
  * agents always attack the nearest opponent with the L5 fight policy;
  * opponents use the L4 fight policy, or escape with probability 1 − `HIER_OPP_FIGHT_RATIO`;
  * Fig. 7a counts per-aircraft-type kills (k), losses (d) and friendly kills (fk) in 2v2. The original
    `evaluation.py` does not compute these.
* **Fig. 8b, escape:** all agents run the escape policy and opponents use the L5 fight policy. There is no
  original script; the categories are:
  * *Killed*: at least one agent destroyed, out-of-bounds included;
  * *Kills*: otherwise, at least one opponent destroyed;
  * *Escaped*: otherwise.
* **Fig. 10b, commander:**
  * the commander policy picks deterministic actions (`explore=False`);
  * both teams use L5 fight;
  * with `RLLIB_FAITHFUL` the GRU state is reset every commander step and chained across agents, as in
    `evaluation.py`.

Outcomes follow the env's info:
* win = all opponents dead before the horizon;
* loss = all agents dead before the horizon;
* draw = horizon reached with both teams alive.

In [ ]:
def make_hl_eval(n, kind, cmd_family=None, n_episodes=None):
    env = Env('high', n=n, m=n)
    reset_b, step_b = make_batched(env)
    E = n_episodes or EVAL_EPISODES

    def run(llp, cmd_params, key):
        k0, k1 = jax.random.split(key)
        s, obs = reset_b(jax.random.split(k0, E))

        def actions(obs, h):
            if kind == 'lowlevel':
                return jnp.ones((E, n), jnp.int32), h
            if kind == 'escape':
                return jnp.zeros((E, n), jnp.int32), h
            zc = jnp.zeros((E, 105))
            if RLLIB_FAITHFUL:    # evaluation.py: state zeroed every step, chained agent -> agent
                hc, acts = jnp.zeros((E, 2, H_CMD)), []
                for i in range(n):
                    lg, _, hc = cmd_step(cmd_family, cmd_params, obs[:, i], zc, hc)
                    acts.append(jnp.argmax(lg, -1))
                return jnp.stack(acts, 1).astype(jnp.int32), h
            lg, _, h2 = cmd_step(cmd_family, cmd_params, obs, jnp.zeros((E, n, 105)), h)
            return jnp.argmax(lg, -1).astype(jnp.int32), h2

        def cond(c):
            return (~jnp.all(c[2])) & (c[5] < HORIZON_HL + 20)

        def body(c):
            s, obs, done, h, key, it = c
            key, k = jax.random.split(key)
            a, h = actions(obs, h)
            s2, obs2, r, d, info, _ = step_b(s, a, jax.random.split(k, E), llp)
            return (tree_where(done, s, s2), jnp.where(done[:, None, None], obs, obs2), done | d, h, key, it + 1)
        c = (s, obs, jnp.zeros(E, bool), jnp.zeros((E, n, 2, H_CMD)), k1, 0)
        s = jax.lax.while_loop(cond, body, c)[0]
        na, no = s['alive'][:, :n].sum(1), s['alive'][:, n:].sum(1)
        H = env.horizon
        win, loss = (no == 0) & (s['steps'] < H), (na == 0) & (s['steps'] < H)
        draw = (s['steps'] >= H) & (na > 0) & (no > 0)
        killed = s['st_d'].sum(1) > 0          # any agent destroyed (opponent, friendly fire or out-of-bounds)
        kills = ~killed & (s['st_opp_by_ag'] > 0)
        return dict(win=win.mean(), loss=loss.mean(), draw=draw.mean(), episodes=jnp.float32(E),
                    k1=s['st_k'][:, 0].sum(), k2=s['st_k'][:, 1].sum(), d1=s['st_d'][:, 0].sum(),
                    d2=s['st_d'][:, 1].sum(), fk1=s['st_fk'][:, 0].sum(), fk2=s['st_fk'][:, 1].sum(),
                    draws=draw.sum(), esc_killed=killed.mean(), esc_kills=kills.mean(),
                    esc_escaped=1.0 - killed.mean() - kills.mean(), ep_len=s['steps'].mean())
    return jax.jit(run)


def have_policy(*names):
    return all(os.path.exists(os.path.join(DIRS['policies'], nm + '.pkl')) for nm in names)


def run_evaluations():
    results = {}
    key = jax.random.PRNGKey(SEED + 777)
    jobs = []
    if have_policy('fight_L5', 'fight_L4', ESC_FOR_HL):
        jobs.append(('fig7_lowlevel', 'lowlevel', None, ll_policies('fight_L5', 'fight_L4')))
    if have_policy('fight_L5', ESC_FOR_HL):
        jobs.append(('fig8b_escape', 'escape', None, ll_policies('fight_L5')))
    if have_policy('commander_gru', 'fight_L5', ESC_FOR_HL):
        jobs.append(('fig10b_commander', 'commander', ('cmd_gru', load_policy('commander_gru')), ll_policies('fight_L5')))
    for name, kind, cmd, llp in jobs:
        res = {}
        for n in EVAL_SIZES:
            t0 = time.time()
            ev = make_hl_eval(n, kind, cmd[0] if cmd else None)
            out = jax.device_get(ev(llp, cmd[1] if cmd else {}, jax.random.fold_in(key, n)))
            res[f'{n}v{n}'] = {k: float(v) for k, v in out.items()}
            r = res[f'{n}v{n}']
            print(f'{name} {n}v{n}: win {r["win"]:.3f} loss {r["loss"]:.3f} draw {r["draw"]:.3f} | '
                  f'escaped {r["esc_escaped"]:.3f} killed {r["esc_killed"]:.3f} kills {r["esc_kills"]:.3f} | '
                  f'{time.time() - t0:.1f}s')
        results[name] = res
        with open(os.path.join(DIRS['eval'], name + '.json'), 'w') as f:
            json.dump(res, f, indent=1)
    if not jobs:
        print('No trained policies found for evaluation (train fsp/fig8/fig10 first).')
    return results


if 'eval' in EXPERIMENTS and not STOPPED:
    EVAL_RESULTS = run_evaluations()

## 7. Figures
Paper-style versions of Fig. 6, 7a, 7b, 8a, 8b, 10a and 10b, saved to `OUT_DIR/figures`. Each figure overlays the
paper's values (black markers or hatched bars).
* Learning curves were read by eye from the PDF (about ±0.2 reward).
* Bar values were measured from the PDF (about ±1 pp); see `docs/paper_results.md`.
* Curves show the mean episode return summed over the team, against env steps. A smoothed line (EMA, like
  TensorBoard) is drawn over the faint raw curve.

The section also plots a trajectory and writes a GIF/MP4 animation of one recorded episode.

In [ ]:
# ---- paper targets (arXiv 2309.11247, digitized; x in millions of env steps) ----------------------------------------
PAPER = {
    'fig6': {'SA-Net': [(1, -2.0), (8, -1.3), (14, -0.7), (19.5, 0.0), (25.5, 1.05), (27.5, 0.95)],
             'SA-Net (no Curr)': [(1, -2.0), (11.5, 0.0), (16, 0.1), (20, 0.2), (27.5, 0.6)],
             'FC': [(4, -1.8), (5, -1.0), (12, -0.7), (20, -0.5), (27.5, 0.03)]},
    'fig7a': {'k-1': 730, 'k-2': 232, 'd-1': 196, 'd-2': 481, 'fk-1': 50, 'fk-2': 39, 'Draw': 162},
    'fig7b': {'2v2': (71.4, 10.1, 18.5), '3v3': (65.3, 10.9, 23.8), '4v4': (56.3, 13.9, 29.8), '5v5': (47.3, 16.1, 36.6)},
    'fig8a': [(0.2, -3.0), (1.15, 0.0), (2.0, 1.0), (2.7, -0.5), (3.5, 3.0), (5.0, 4.0), (5.8, 5.0), (7.0, 5.0),
              (8.2, 5.3)],
    'fig8b': {'2v2': (79.4, 15.9, 4.7), '3v3': (58.2, 33.1, 8.7), '4v4': (31.1, 52.2, 16.7), '5v5': (14.2, 67.2, 18.7)},
    'fig10a': {'GRU-Net': [(0.05, -3.3), (1.1, 1.0), (3.0, 0.9), (6.7, 0.85)],
               'SA-Net': [(0.05, -3.0), (1.5, 0.0), (4.0, 0.1), (6.7, 0.2)],
               'FC': [(0.05, -2.6), (1.5, -0.5), (4.0, -0.35), (6.7, -0.2)]},
    'fig10b': {'2v2': (51.2, 29.1, 19.7), '3v3': (46.0, 28.1, 25.9), '4v4': (35.1, 26.1, 38.8), '5v5': (23.1, 20.9, 56.0)},
}
COLORS = {'a': '#1f77b4', 'b': '#ff7f0e', 'c': '#2ca02c', 'win': '#2ca02c', 'loss': '#d62728', 'draw': '#7f7f7f'}
FIG_DIR = DIRS['figures']


def read_log(stage):
    path = os.path.join(DIRS['logs'], stage + '.csv')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return None
    x = np.array([float(r['env_steps']) for r in rows]) / 1e6
    y = np.array([float(r['ep_return']) for r in rows])
    ok = np.isfinite(y)
    return x[ok], y[ok]


def ema(y, alpha=None):
    if len(y) == 0:
        return y
    alpha = alpha or min(0.3, 20.0 / len(y))
    out, m = np.empty_like(y), y[0]
    for i, v in enumerate(y):
        m = (1 - alpha) * m + alpha * v
        out[i] = m
    return out


def curve_fig(fname, title, curves, paper, ylabel='Mean episode reward (team)'):
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for (label, stage), c in zip(curves, (COLORS['a'], COLORS['b'], COLORS['c'])):
        d = read_log(stage)
        if d is not None and len(d[0]):
            ax.plot(d[0], d[1], color=c, alpha=0.2, lw=0.8)
            ax.plot(d[0], ema(d[1]), color=c, lw=2, label=f'{label} (ours)')
        if label in paper:
            px, py = zip(*paper[label])
            ax.plot(px, py, color=c, ls='--', marker='o', ms=4, mfc='white', lw=1, label=f'{label} (paper)')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Env steps [M]'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, fname), dpi=140); plt.close(fig)


def load_eval(name):
    path = os.path.join(DIRS['eval'], name + '.json')
    return json.load(open(path)) if os.path.exists(path) else None


def outcome_bars(fname, title, ours, paper, keys, labels):
    scen = list(paper.keys())
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    width = 0.13
    for j, (k, lab) in enumerate(zip(keys, labels)):
        c = COLORS.get(lab.lower(), ['#2ca02c', '#d62728', '#1f77b4'][j])
        xs = np.arange(len(scen)) + (j - 1) * 2.2 * width
        pv = [paper[s][j] for s in scen]
        ax.bar(xs - width / 2, pv, width, color='white', edgecolor=c, hatch='///', label=f'{lab} (paper)')
        if ours:
            ov = [100 * ours[s][k] if s in ours else np.nan for s in scen]
            ax.bar(xs + width / 2, ov, width, color=c, label=f'{lab} (ours)')
            for x_, v in zip(xs + width / 2, ov):
                if np.isfinite(v):
                    ax.text(x_, v + 1, f'{v:.0f}', ha='center', fontsize=6)
    ax.set_xticks(np.arange(len(scen))); ax.set_xticklabels([s.replace('v', ' vs ') for s in scen])
    ax.set_ylabel('% of episodes'); ax.set_title(title); ax.set_ylim(0, 100)
    ax.legend(fontsize=7, ncol=2); ax.grid(axis='y', alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, fname), dpi=140); plt.close(fig)


def make_figures():
    curve_fig('fig6_fight_L3.png', 'Fig. 6: fight policy at L3',
              [('SA-Net', 'fight_L3'), ('SA-Net (no Curr)', 'fight_L3_nocurr'), ('FC', 'fight_L3_fc')], PAPER['fig6'])
    curve_fig('fig8a_escape_L3.png', 'Fig. 8a: escape policy at L3', [('Escape', 'escape_L3')],
              {'Escape': PAPER['fig8a']})
    curve_fig('fig10a_commander.png', 'Fig. 10a: commander (3 vs 3)',
              [('GRU-Net', 'commander_gru'), ('SA-Net', 'commander_sa'), ('FC', 'commander_fc')], PAPER['fig10a'])
    ev7 = load_eval('fig7_lowlevel')
    outcome_bars('fig7b_eval_fight.png', 'Fig. 7b: L5 fight vs L4 fight (low level only)', ev7, PAPER['fig7b'],
                 ['win', 'loss', 'draw'], ['Win', 'Loss', 'Draw'])
    ev8 = load_eval('fig8b_escape')
    outcome_bars('fig8b_eval_escape.png', 'Fig. 8b: escape evaluation', ev8, PAPER['fig8b'],
                 ['esc_escaped', 'esc_killed', 'esc_kills'], ['Escaped', 'Killed', 'Kills'])
    ev10 = load_eval('fig10b_commander')
    outcome_bars('fig10b_eval_commander.png', 'Fig. 10b: commander evaluation', ev10, PAPER['fig10b'],
                 ['win', 'loss', 'draw'], ['Win', 'Loss', 'Draw'])
    # Fig. 7a: per-type counts per 1000 episodes (2 vs 2)
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    cats = list(PAPER['fig7a'].keys())
    xs = np.arange(len(cats))
    ax.bar(xs - 0.2, [PAPER['fig7a'][c] for c in cats], 0.4, color='white', edgecolor='k', hatch='///', label='paper')
    if ev7 and '2v2' in ev7:
        r = ev7['2v2']
        sc = 1000.0 / r['episodes']
        ours = [r['k1'], r['k2'], r['d1'], r['d2'], r['fk1'], r['fk2'], r['draws']]
        ax.bar(xs + 0.2, [v * sc for v in ours], 0.4, color=COLORS['a'], label='ours')
    ax.set_xticks(xs); ax.set_xticklabels(cats); ax.set_ylabel('count per 1000 episodes')
    ax.set_title('Fig. 7a: L5 vs L4, 2 vs 2 (k kills, d destroyed, fk friendly kills; -1/-2 = AC type)', fontsize=9)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'fig7a_eval_types.png'), dpi=140); plt.close(fig)
    print('figures written to', FIG_DIR, sorted(os.listdir(FIG_DIR)))


# ---- trajectories and animation ---------------------------------------------------------------------------------------
KM_LON, KM_LAT = 111.32 * math.cos(5.25 * D2R), 110.6


def record_episode(kind='commander', seed=0, max_steps=600):
    """Record one 3v3 commander-env episode (commander GRU if trained, else 'attack nearest') or, if the L5/L4 policies
    are missing, one low-level L3 episode with the best available fight policy. Returns frames + metadata."""
    key = jax.random.PRNGKey(SEED + 1000 + seed)
    frames, meta = [], {}
    if have_policy('fight_L5', ESC_FOR_HL):
        env = Env('high', n=3, m=3, record=True)
        llp = ll_policies('fight_L5')
        cmd = load_policy('commander_gru') if have_policy('commander_gru') else None
        step = jax.jit(env.step)
        s, obs = jax.jit(env.reset)(key)
        h = jnp.zeros((3, 2, H_CMD))
        meta['title'] = '3 vs 3, ' + ('commander GRU + L5 low-level policies' if cmd else 'L5 fight, attack nearest')
        for t in range(max_steps):
            key, k = jax.random.split(key)
            if cmd is not None:
                lg, _, h = cmd_step('cmd_gru', cmd, obs, jnp.zeros((3, 105)), h)
                a = jnp.argmax(lg, -1)
            else:
                a = jnp.ones((3,), jnp.int32)
            s, obs, r, d, info, (tr, act) = step(s, a, k, llp)
            tr, act = jax.device_get((tr, act))
            for i in np.nonzero(act)[0]:
                frames.append({kk: v[i] for kk, v in tr.items()})
            if bool(d):
                break
        meta['n'] = 3
    else:
        stage = next((n for n in ('fight_L3', 'fight_L3_nocurr', 'fight_L2', 'fight_L1') if have_policy(n)), None)
        env = Env('low', level=3, mode='fight', record=True)
        params = load_policy(stage) if stage else low_init('fight_sa', key)
        step = jax.jit(env.step)
        s, obs = jax.jit(env.reset)(key)
        meta['title'] = f'2 vs 2 at L3, policy {stage or "untrained"}'
        for t in range(max_steps):
            key, k = jax.random.split(key)
            a = jnp.stack([md_argmax(low_actor_infer('fight_sa', params, 1, obs[0]), SPLITS1),
                           md_argmax(low_actor_infer('fight_sa', params, 2, obs[1]), SPLITS2)])
            s, obs, r, d, info, tr = step(s, a, k, ())
            frames.append(jax.device_get(tr))
            if bool(d):
                break
        meta['n'] = 2
    info = jax.device_get(info)
    meta['outcome'] = 'win' if info['win'] else ('loss' if info['loss'] else ('draw' if info['draw'] else 'end'))
    meta['ms'] = env.ms
    return frames, meta


def _xy(lat, lon):
    return lon * KM_LON, lat * KM_LAT


def plot_trajectories(frames, meta, path):
    n = meta['n']
    lat = np.array([f['lat'] for f in frames]); lon = np.array([f['lon'] for f in frames])
    alive = np.array([f['alive'] for f in frames])
    fig, ax = plt.subplots(figsize=(6, 6))
    for i in range(lat.shape[1]):
        c = '#1f77b4' if i < n else '#d62728'
        last = np.nonzero(alive[:, i])[0]
        upto = (last[-1] + 1) if len(last) else 1
        x, y = _xy(lat[:upto, i], lon[:upto, i])
        ax.plot(x, y, color=c, lw=1.4, label=('agents' if i == 0 else 'opponents' if i == n else None))
        ax.plot(x[0], y[0], 'o', color=c, ms=4)
        ax.plot(x[-1], y[-1], 'x' if upto < len(frames) else '>', color=c, ms=8)
    rl, ro, ra = (np.array([f[k] for f in frames]) for k in ('r_lat', 'r_lon', 'r_alive'))
    for j in range(rl.shape[1]):
        x, y = _xy(np.where(ra[:, j], rl[:, j], np.nan), np.where(ra[:, j], ro[:, j], np.nan))
        ax.plot(x, y, ':', color='#1f77b4' if j < n else '#d62728', lw=1)
    L = meta['ms'] * KM_LAT
    ax.set_xlim(0, meta['ms'] * KM_LON); ax.set_ylim(0, L); ax.set_aspect('equal')
    ax.set_xlabel('east [km]'); ax.set_ylabel('north [km]')
    ax.set_title(f"{meta['title']}\n{len(frames)} ticks, outcome: {meta['outcome']} (x = destroyed, dotted = rockets)",
                 fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(path, dpi=140); plt.close(fig)


def animate(frames, meta, path_noext, max_frames=240, fps=15):
    from matplotlib import animation
    n = meta['n']
    idx = np.unique(np.linspace(0, len(frames) - 1, min(max_frames, len(frames))).astype(int))
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(0, meta['ms'] * KM_LON); ax.set_ylim(0, meta['ms'] * KM_LAT); ax.set_aspect('equal')
    ax.set_xlabel('east [km]'); ax.set_ylabel('north [km]'); ax.grid(alpha=0.3)
    T = len(frames[0]['lat'])
    cols = ['#1f77b4' if i < n else '#d62728' for i in range(T)]
    trails = [ax.plot([], [], color=cols[i], lw=1)[0] for i in range(T)]
    heads = ax.scatter(np.zeros(T), np.zeros(T), c=cols, s=30, marker='o')
    rk = ax.scatter(np.zeros(T), np.zeros(T), c=cols, s=12, marker='*')
    txt = ax.set_title('')

    def upd(k):
        f = frames[idx[k]]
        x, y = _xy(f['lat'], f['lon'])
        heads.set_offsets(np.c_[np.where(f['alive'], x, np.nan), np.where(f['alive'], y, np.nan)])
        rx, ry = _xy(f['r_lat'], f['r_lon'])
        rk.set_offsets(np.c_[np.where(f['r_alive'], rx, np.nan), np.where(f['r_alive'], ry, np.nan)])
        for i in range(T):
            hist = [frames[j] for j in range(max(0, idx[k] - 60), idx[k] + 1) if frames[j]['alive'][i]]
            if hist:
                hx, hy = _xy(np.array([h['lat'][i] for h in hist]), np.array([h['lon'][i] for h in hist]))
                trails[i].set_data(hx, hy)
            else:
                trails[i].set_data([], [])
        txt.set_text(f"{meta['title']} | tick {idx[k] + 1}")
        txt.set_fontsize(8)
        return trails + [heads, rk, txt]
    ani = animation.FuncAnimation(fig, upd, frames=len(idx), blit=False)
    out = []
    try:
        ani.save(path_noext + '.gif', writer=animation.PillowWriter(fps=fps)); out.append(path_noext + '.gif')
    except Exception as e:
        print('GIF export failed:', e)
    if animation.writers.is_available('ffmpeg'):
        try:
            ani.save(path_noext + '.mp4', writer=animation.FFMpegWriter(fps=fps)); out.append(path_noext + '.mp4')
        except Exception as e:
            print('MP4 export failed:', e)
    plt.close(fig)
    return out


if 'figures' in EXPERIMENTS:
    make_figures()
    _frames, _meta = record_episode()
    plot_trajectories(_frames, _meta, os.path.join(FIG_DIR, 'trajectory_episode.png'))
    print('animation:', animate(_frames, _meta, os.path.join(FIG_DIR, 'episode')))
    print(_meta['title'], '->', _meta['outcome'], f'({len(_frames)} ticks)')

## 8. Known deviations from the original (what "faithful" means here)

The environment logic, rewards, observations, networks and PPO hyper-parameters follow the original code
line by line, including its quirks. Distributional parity was checked against the original Python env, with
random agents vs the scripted L1–L3 opponents (2000 episodes each). Win/loss/draw rates, team return, episode length,
deaths and rocket launches agree within sampling noise. What differs:

1. **RNG streams.** The original uses Python `random` and an OS-seeded `random.Random` per reset, which is not reproducible
   anyway. Here every draw site has its own JAX key, so only the *distributions* match.
2. **Geodesics.** Instead of `geographiclib`, a local ellipsoidal flat-earth model is used. Its errors are below
   0.2 m in distance, below 0.001° in bearing and below 1 cm per move, all far below the weapon thresholds. Positions
   are float32 offsets from (5°N, 7°E).
3. **Batch geometry.** Rollouts have a fixed length per env with auto-reset, and bootstrap V at the cut. The original
   uses RLlib `complete_episodes`, so its batches slightly overshoot 2000/1000 steps and never cut episodes. Minibatches
   have a fixed size (the last partial minibatch of each epoch is dropped; a new shuffle is drawn every epoch).
4. **RLlib sequence chunks.** The 20-step chunks used for the training-time attention (`RLLIB_FAITHFUL`) and for
   the GRU are aligned to the rollout grid, and split at episode boundaries. RLlib aligns them to the start of each
   episode. RLlib's zero padding is emulated analytically (pad keys with multiplicity, pad rows in the loss), not by
   materializing padded rows.
5. **Paper-only baselines.** The FC baselines (Fig. 6/10) and the SA commander (Fig. 10) are not in the released
   code. They are reconstructed from the paper's description: 2×500 tanh FC with a centralized FC critic; the
   commander with MHA(200) in place of the GRU.
6. **Missing original scripts.**
   * Fig. 7a per-type counts and the Fig. 8b escape evaluation have no original script. Their category definitions
     are given in Section 6.
   * The FSP opponents are the parameters at the *end* of the previous stage. The original exports a snapshot every
     50 iterations and uses the last one.
7. **Not ported.** `glob_frac` reward sharing (default 0) is not implemented. The original's crash on length-1
   episodes in the critic callback is not reproduced. Rendering uses matplotlib instead of cairo/cartopy.
8. **Framework numerics.** JAX vs PyTorch: float32 matmul order, Adam epsilon placement (identical formula),
   orthogonal initializers drawn from a different RNG.
9. **Training budgets.** Not stated in the paper; `full` uses about 10k RLlib iterations per level and the paper's axis
   lengths. The `throughput` preset changes the sample reuse (fewer SGD steps per env step), so its learning curves
   per env step are not directly comparable with the paper.
10. **Curriculum transfer.** Like `algo.restore`, the transfer copies params, Adam state and the KL coefficient, but
    the env-step counter restarts at 0 for each stage, which matches the paper's x-axes.